# 成员B：TLCC、回归建模、预测比较与显著性检验

项目：**情绪与现实的博弈：基于多模态数据的科技劳动力市场前瞻信息研究**

本 Notebook 对应成员B负责的模型分析、机制检验和显著性检验部分，围绕三个研究问题展开：

- **RQ1**：新闻情绪和裁员相关新闻强度是领先于科技裁员事件，还是滞后于实际裁员？
- **RQ2**：在控制美国宏观劳动力指标后，新闻情绪特征是否能提升下一周裁员事件数预测？
- **RQ3**：AI 公司与非 AI 公司裁员事件对新闻情绪的响应是否不同？

质量控制原则：不随机打乱时间序列；不把 `target_next_week_*` 等未来目标列放入特征；不删除零裁员周；主目标优先使用下一周裁员事件数而非裁员人数；可选模型失败时不影响主流程。


## 第 0 部分：环境准备与依赖导入

本部分设置项目路径、创建输出目录、导入依赖，并检查可选模型包是否可用。所有随机模型统一使用 `random_state=42`，保证结果可复现。


In [1]:
# ==============================
# 第 0 部分：环境准备与依赖导入
# ==============================

from pathlib import Path
import warnings
import json
import math
import traceback
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# seaborn 是可选依赖；如果存在，则只用于简单风格设置。
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

# sklearn：时间序列切分、Pipeline、预处理、模型与指标。
from sklearn.model_selection import TimeSeriesSplit, train_test_split  # train_test_split 不用于随机时间切分，仅保留以备工具函数需要。
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, Ridge, LassoCV, Lasso, PoissonRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, mean_poisson_deviance

# statsmodels：HAC 稳健回归、Poisson / Negative Binomial GLM。
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan

# scipy：相关性显著性检验。
from scipy import stats

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 项目根目录与输出目录。注意：本 Notebook 只读取原始数据，不修改原始数据文件。
BASE_DIR = Path("/Users/peiyifan/Desktop/机器学习")
FIG_DIR = BASE_DIR / "03_figures"
RESULT_DIR = BASE_DIR / "04_results"
TEXT_DIR = BASE_DIR / "05_report_text"
CODE_DIR = BASE_DIR / "02_code"

for d in [CODE_DIR, FIG_DIR, RESULT_DIR, TEXT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 可选：XGBoost。如果没有安装，后续自动跳过，不中断 Notebook。
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception as e:
    XGBRegressor = None
    XGBOOST_AVAILABLE = False
    XGBOOST_IMPORT_ERROR = str(e)

# 可选：TensorFlow / Keras。只有样本量足够时才考虑 LSTM。
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    TENSORFLOW_AVAILABLE = True
except Exception as e:
    tf = None
    TENSORFLOW_AVAILABLE = False
    TENSORFLOW_IMPORT_ERROR = str(e)

print("项目根目录：", BASE_DIR)
print("图表目录：", FIG_DIR)
print("结果目录：", RESULT_DIR)
print("文字目录：", TEXT_DIR)
print("XGBoost 可用：", XGBOOST_AVAILABLE)
print("TensorFlow 可用：", TENSORFLOW_AVAILABLE)


项目根目录： /Users/peiyifan/Desktop/机器学习
图表目录： /Users/peiyifan/Desktop/机器学习/03_figures
结果目录： /Users/peiyifan/Desktop/机器学习/04_results
文字目录： /Users/peiyifan/Desktop/机器学习/05_report_text
XGBoost 可用： False
TensorFlow 可用： False


## 通用工具函数

本单元集中放置后续反复使用的工具函数，包括显著性星号、特征去重、评价指标、结果写入、报告文本拼接等。这样可以减少各章节重复代码，并提高容错性。


In [2]:
# ==============================
# 通用工具函数
# ==============================

def safe_write_text(path, text):
    """安全写入 Markdown / 文本文件。"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(str(text), encoding="utf-8")


def sig_star(p):
    """根据 p 值返回显著性星号。"""
    if pd.isna(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def clean_feature_list(cols, df_columns=None):
    """去重、保序，并只保留 DataFrame 中存在的特征。"""
    seen = set()
    out = []
    for c in cols:
        if c is None or c in seen:
            continue
        if df_columns is not None and c not in df_columns:
            continue
        seen.add(c)
        out.append(c)
    return out


def is_forbidden_feature(col):
    """判断某列是否明显包含未来信息或标签信息，禁止进入 X。

    修复说明：旧版本把 "y_" 作为任意位置的禁用关键词，导致 weekly_ 开头的
    核心变量被误删。这里仅禁止真正名为 y 或以 y_ 开头的标签列。
    """
    low = str(col).lower()
    forbidden_keywords = [
        "target_next_week",
        "next_week",
        "future",
        "lead",
        "t_plus",
        "label",
    ]
    if low == "y" or low.startswith("y_"):
        return True
    return any(k in low for k in forbidden_keywords)


def numeric_existing_features(df, features):
    """只保留存在且可转为数值的特征列。"""
    valid = []
    for c in clean_feature_list(features, df.columns):
        if is_forbidden_feature(c):
            continue
        if pd.api.types.is_numeric_dtype(df[c]) or pd.to_numeric(df[c], errors="coerce").notna().sum() > 0:
            valid.append(c)
    return valid


def smape(y_true, y_pred):
    """计算 sMAPE，避免 0 值导致普通 MAPE 失真。"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom > 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(2 * np.abs(y_pred[mask] - y_true[mask]) / denom[mask])


def directional_accuracy(y_true, y_pred):
    """比较相邻测试期真实值和预测值的增减方向是否一致。"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2:
        return np.nan
    true_diff = np.diff(y_true)
    pred_diff = np.diff(y_pred)
    return np.mean(np.sign(true_diff) == np.sign(pred_diff))


def evaluate_predictions(y_true, y_pred):
    """统一计算样本外预测指标。"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_pred = np.clip(y_pred, 0, None)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    nonzero = y_true > 0
    mape_nonzero = mean_absolute_percentage_error(y_true[nonzero], y_pred[nonzero]) if nonzero.sum() > 0 else np.nan
    smape_val = smape(y_true, y_pred)
    da = directional_accuracy(y_true, y_pred)
    try:
        poisson_dev = mean_poisson_deviance(np.clip(y_true, 0, None), np.clip(y_pred, 1e-9, None))
    except Exception:
        poisson_dev = np.nan
    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE_nonzero": mape_nonzero,
        "sMAPE": smape_val,
        "Poisson_Deviance": poisson_dev,
        "Directional_Accuracy": da,
    }


def prepare_X_y(df, features, target):
    """构造模型输入 X 和标签 y，并把特征尽量转为数值。"""
    features = numeric_existing_features(df, features)
    X = df[features].copy()
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    y = pd.to_numeric(df[target], errors="coerce")
    mask = y.notna()
    return X.loc[mask], y.loc[mask], features, mask


def plot_save_close(path, dpi=180):
    """统一保存图表并关闭画布。"""
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()


def describe_available(items):
    """把变量列表转成适合报告阅读的短文本。"""
    if not items:
        return "无可用变量"
    return "、".join(items[:12]) + (" 等" if len(items) > 12 else "")

print("通用工具函数已定义。")


通用工具函数已定义。


## 第 1 部分：读取数据与基础检查

本部分自动按优先级寻找成员A生成的建模矩阵，识别时间列，按时间排序，检查目标变量。如果没有现成的 next-week 标签但存在当前周事件数，则用 `shift(-1)` 生成下一周目标。


In [3]:
# ==============================
# 第 1 部分：读取数据与基础检查
# ==============================

candidate_files = [
    BASE_DIR / "final_model_matrix_v1.csv",
    BASE_DIR / "final_model_matrix.csv",
    BASE_DIR / "final_weekly_panel.csv",
    BASE_DIR / "final_weekly_panel_v0.csv",
]

selected_file = None
for p in candidate_files:
    if p.exists():
        selected_file = p
        break

if selected_file is None:
    msg = (
        "未找到建模数据文件。请先运行成员A的数据 pipeline，生成以下任一文件：\n"
        + "\n".join([str(p) for p in candidate_files])
    )
    safe_write_text(TEXT_DIR / "memberB_data_check_notes.md", "# 数据检查说明\n\n" + msg)
    raise FileNotFoundError(msg)

# 读取 CSV。low_memory=False 可以减少混合类型列的警告。
df = pd.read_csv(selected_file, low_memory=False)

# 识别时间列。
time_candidates = ["week", "date", "Week", "week_start", "start_week"]
time_col = next((c for c in time_candidates if c in df.columns), None)
if time_col is None:
    raise ValueError(f"未找到时间列。候选列为：{time_candidates}")

# 转换时间列并按时间升序排序，保证所有滞后与切分都遵循时间顺序。
df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
df = df.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)

# 尽量把明显的数值列转为数值；无法转换的保留原样。
for c in df.columns:
    if c == time_col:
        continue
    if df[c].dtype == "object":
        converted = pd.to_numeric(df[c], errors="coerce")
        if converted.notna().sum() >= max(1, int(0.5 * df[c].notna().sum())):
            df[c] = converted

# 检查主目标。如果没有 next-week 目标，但有当前周事件数，则使用 shift(-1) 构造。
main_target = "target_next_week_layoff_event_count"
main_log_target = "target_next_week_log_layoff_event_count"

if main_target not in df.columns:
    if "weekly_layoff_event_count" in df.columns:
        df[main_target] = pd.to_numeric(df["weekly_layoff_event_count"], errors="coerce").shift(-1)
        print("未发现 target_next_week_layoff_event_count，已由 weekly_layoff_event_count.shift(-1) 生成。")
    else:
        raise ValueError("缺少主目标 target_next_week_layoff_event_count，且无法由 weekly_layoff_event_count 生成。")

if main_log_target not in df.columns:
    df[main_log_target] = np.log1p(pd.to_numeric(df[main_target], errors="coerce"))

# 删除没有下一周标签的记录，通常是最后一周。注意：不删除零裁员周。
df[main_target] = pd.to_numeric(df[main_target], errors="coerce")
df[main_log_target] = pd.to_numeric(df[main_log_target], errors="coerce")
df = df.dropna(subset=[main_target]).reset_index(drop=True)

# 基础检查输出。
missing_rate = df.isna().mean().sort_values(ascending=False).head(30)
summary_rows = [
    {"item": "selected_file", "value": str(selected_file)},
    {"item": "n_rows", "value": df.shape[0]},
    {"item": "n_columns", "value": df.shape[1]},
    {"item": "time_col", "value": time_col},
    {"item": "time_min", "value": df[time_col].min()},
    {"item": "time_max", "value": df[time_col].max()},
    {"item": "main_target_exists", "value": main_target in df.columns},
    {"item": "main_log_target_exists", "value": main_log_target in df.columns},
]
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULT_DIR / "memberB_data_check_summary.csv", index=False, encoding="utf-8-sig")

missing_df = missing_rate.reset_index()
missing_df.columns = ["variable", "missing_rate"]
missing_df.to_csv(RESULT_DIR / "memberB_missing_rate_top30.csv", index=False, encoding="utf-8-sig")

notes = f"""# 成员B数据检查说明

- 读取文件：`{selected_file}`
- 时间列：`{time_col}`
- 样本量：{df.shape[0]}
- 变量数：{df.shape[1]}
- 时间范围：{df[time_col].min().date()} 至 {df[time_col].max().date()}
- 主目标：`{main_target}`
- log 主目标：`{main_log_target}`

## 缺失率最高的前 30 个变量

{missing_df.to_markdown(index=False)}

说明：本 Notebook 不删除零裁员周；仅删除无法获得下一周标签的记录。
"""
safe_write_text(TEXT_DIR / "memberB_data_check_notes.md", notes)

print("数据 shape：", df.shape)
print("时间范围：", df[time_col].min(), "至", df[time_col].max())
print("列名：")
print(list(df.columns))
print("缺失率最高的前 30 个变量：")
display(missing_df)


数据 shape： (223, 91)
时间范围： 2022-01-03 00:00:00 至 2026-04-06 00:00:00
列名：
['week', 'weekly_layoff_event_count', 'weekly_known_layoff_count', 'weekly_ai_layoff_event_count', 'weekly_non_ai_layoff_event_count', 'weekly_ai_known_layoff_count', 'weekly_non_ai_known_layoff_count', 'weekly_layoff_companies_count', 'weekly_layoff_countries_count', 'weekly_news_count', 'weekly_layoff_news_count', 'weekly_hiring_news_count', 'weekly_negative_news_count', 'weekly_positive_news_count', 'weekly_neutral_news_count', 'weekly_avg_sentiment', 'weekly_layoff_avg_sentiment', 'weekly_hiring_avg_sentiment', 'weekly_negative_share', 'weekly_layoff_news_share', 'sentiment_shock', 'unemployment_rate', 'jolts_job_openings_k', 'initial_jobless_claims_k', 'information_sector_emp_k', 'computer_math_emp_k', 'financial_emp_k', 'openings_per_unemployed', 'tech_emp_yoy_pct', 'claims_4w_avg', 'global_unemployment_rate_pct_mean', 'global_youth_unemployment_pct_mean', 'global_employment_to_pop_pct_mean', 'has_news_week',

## 第 2 部分：变量分组与防止信息泄漏

本部分动态识别新闻变量、宏观变量、历史裁员变量和 AI/非 AI 异质性变量。任何包含未来目标含义的列都会被排除，尤其是 `target_next_week_*`、`next_week`、`future`、`lead` 等。


In [4]:
# ==============================
# 第 2 部分：变量分组与防止信息泄漏
# ==============================

all_cols = list(df.columns)
all_cols_lower = {c: c.lower() for c in all_cols}

# 新闻变量：优先变量及 lag1-lag4。
base_news = [
    "weekly_news_count",
    "weekly_layoff_news_count",
    "weekly_layoff_news_share",
    "weekly_negative_share",
    "weekly_avg_sentiment",
    "weekly_layoff_avg_sentiment",
    "sentiment_shock",
    "has_news_week",
]
news_priority = []
for b in base_news:
    news_priority.append(b)
    for k in range(1, 5):
        news_priority.append(f"{b}_lag{k}")

news_keywords = ["news", "sentiment", "negative", "layoff_news", "hiring_news"]
news_auto = [c for c in all_cols if any(k in all_cols_lower[c] for k in news_keywords)]
news_features = numeric_existing_features(df, news_priority + news_auto)

# 宏观变量：优先变量 + 关键词匹配，排除新闻变量和未来标签。
macro_priority = [
    "initial_jobless_claims_k",
    "claims_4w_avg",
    "unemployment_rate",
    "jolts_job_openings_k",
    "information_sector_emp_k",
    "computer_math_emp_k",
]
macro_keywords = ["claims", "unemployment", "jolts", "openings", "employment", "emp", "labor", "gdp", "inflation"]
macro_auto = []
for c in all_cols:
    low = all_cols_lower[c]
    if any(k in low for k in macro_keywords) and not any(k in low for k in news_keywords):
        macro_auto.append(c)
macro_features = numeric_existing_features(df, macro_priority + macro_auto)

# 历史裁员变量：只保留 lag 和 rolling，避免误把未来目标放入特征。
ar_priority = [
    "weekly_layoff_event_count_lag1",
    "weekly_layoff_event_count_lag2",
    "weekly_layoff_event_count_lag3",
    "weekly_layoff_event_count_lag4",
    "weekly_layoff_event_count_roll4_mean",
    "weekly_layoff_event_count_roll4_sum",
]
ar_auto = []
for c in all_cols:
    low = all_cols_lower[c]
    if "layoff_event_count_lag" in low or "layoff_event_count_roll" in low:
        ar_auto.append(c)
ar_features = numeric_existing_features(df, ar_priority + ar_auto)

# 如果没有 lag/rolling AR 特征，但存在当前周事件数，则允许在 AR baseline 中作为当前可观测状态使用。
# 注意：这不是未来信息，但解释时应说明它代表预测时已知的当前状态。
if not ar_features and "weekly_layoff_event_count" in df.columns:
    ar_features = numeric_existing_features(df, ["weekly_layoff_event_count"])

# AI / 非 AI 异质性变量。
heterogeneity_priority = [
    "weekly_ai_layoff_event_count",
    "weekly_non_ai_layoff_event_count",
    "weekly_ai_layoff_event_count_lag1",
    "weekly_non_ai_layoff_event_count_lag1",
]
heterogeneity_features = numeric_existing_features(df, heterogeneity_priority)

# 二次防泄漏：所有特征组都排除 forbidden 列。
news_features = [c for c in news_features if not is_forbidden_feature(c)]
macro_features = [c for c in macro_features if not is_forbidden_feature(c)]
ar_features = [c for c in ar_features if not is_forbidden_feature(c)]
heterogeneity_features = [c for c in heterogeneity_features if not is_forbidden_feature(c)]

feature_group_rows = []
for group_name, feats in [
    ("news_features", news_features),
    ("macro_features", macro_features),
    ("ar_features", ar_features),
    ("heterogeneity_features", heterogeneity_features),
]:
    for f in feats:
        feature_group_rows.append({"feature_group": group_name, "feature": f})

feature_group_df = pd.DataFrame(feature_group_rows)
feature_group_df.to_csv(RESULT_DIR / "memberB_feature_groups.csv", index=False, encoding="utf-8-sig")

print("新闻变量数：", len(news_features), describe_available(news_features))
print("宏观变量数：", len(macro_features), describe_available(macro_features))
print("AR 变量数：", len(ar_features), describe_available(ar_features))
print("异质性变量数：", len(heterogeneity_features), describe_available(heterogeneity_features))
display(feature_group_df.head(80))


新闻变量数： 53 weekly_news_count、weekly_news_count_lag1、weekly_news_count_lag2、weekly_news_count_lag3、weekly_news_count_lag4、weekly_layoff_news_count、weekly_layoff_news_count_lag1、weekly_layoff_news_count_lag2、weekly_layoff_news_count_lag3、weekly_layoff_news_count_lag4、weekly_layoff_news_share、weekly_layoff_news_share_lag1 等
宏观变量数： 12 initial_jobless_claims_k、claims_4w_avg、unemployment_rate、jolts_job_openings_k、information_sector_emp_k、computer_math_emp_k、financial_emp_k、openings_per_unemployed、tech_emp_yoy_pct、global_unemployment_rate_pct_mean、global_youth_unemployment_pct_mean、global_employment_to_pop_pct_mean
AR 变量数： 5 weekly_layoff_event_count_lag1、weekly_layoff_event_count_lag2、weekly_layoff_event_count_roll4_mean、log_weekly_layoff_event_count_lag1、log_weekly_layoff_event_count_lag2
异质性变量数： 2 weekly_ai_layoff_event_count、weekly_non_ai_layoff_event_count
             feature_group                                 feature
0            news_features                       weekly_news_count


## 第 3 部分：时间序列训练测试切分

本部分按时间顺序切分训练集和测试集。默认使用 2022-01-03 至 2024-12-31 作为训练期，2025-01-01 至 2026-04-13 作为测试期；如果数据范围不足，则自动改为前 75% / 后 25%。


In [5]:
# ==============================
# 第 3 部分：时间序列训练测试切分
# ==============================

preferred_train_start = pd.Timestamp("2022-01-03")
preferred_train_end = pd.Timestamp("2024-12-31")
preferred_test_start = pd.Timestamp("2025-01-01")
preferred_test_end = pd.Timestamp("2026-04-13")

# 判断默认时间切分是否有足够样本。
train_mask_default = (df[time_col] >= preferred_train_start) & (df[time_col] <= preferred_train_end)
test_mask_default = (df[time_col] >= preferred_test_start) & (df[time_col] <= preferred_test_end)

if train_mask_default.sum() >= 20 and test_mask_default.sum() >= 5:
    train_df = df.loc[train_mask_default].copy()
    test_df = df.loc[test_mask_default].copy()
    split_method = "calendar_default"
else:
    split_idx = int(len(df) * 0.75)
    split_idx = max(1, min(split_idx, len(df) - 1))
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()
    split_method = "fallback_75_25"

split_summary = pd.DataFrame([
    {
        "dataset": "train",
        "split_method": split_method,
        "n_obs": len(train_df),
        "time_min": train_df[time_col].min(),
        "time_max": train_df[time_col].max(),
        "target_mean": train_df[main_target].mean(),
        "target_std": train_df[main_target].std(),
    },
    {
        "dataset": "test",
        "split_method": split_method,
        "n_obs": len(test_df),
        "time_min": test_df[time_col].min(),
        "time_max": test_df[time_col].max(),
        "target_mean": test_df[main_target].mean(),
        "target_std": test_df[main_target].std(),
    },
])
split_summary.to_csv(RESULT_DIR / "memberB_train_test_split_summary.csv", index=False, encoding="utf-8-sig")

print("切分方式：", split_method)
display(split_summary)


切分方式： calendar_default
  dataset      split_method  n_obs   time_min   time_max  target_mean  target_std
0   train  calendar_default    157 2022-01-03 2024-12-30    11.668790   13.947883
1    test  calendar_default     66 2025-01-06 2026-04-06     3.575758    3.167433


## 第 4 部分：RQ1 机制检验一：TLCC 时滞交叉相关

本部分计算新闻变量在 `t-k` 周与裁员事件数在 `t` 周之间的相关系数。这里定义：`lag=1` 表示新闻变量领先裁员事件 1 周，即 `corr(news_{t-1}, layoff_t)`；`lag<0` 表示新闻变量滞后于裁员事件。


In [6]:
# ==============================
# 第 4 部分：RQ1 TLCC 时滞交叉相关
# ==============================

# 目标变量：优先当前周裁员事件数。如果没有，则使用下一周主目标近似，并在报告中说明。
if "weekly_layoff_event_count" in df.columns:
    tlcc_target = "weekly_layoff_event_count"
    tlcc_target_note = "使用 weekly_layoff_event_count 作为 TLCC 目标变量。"
else:
    tlcc_target = main_target
    tlcc_target_note = "未找到 weekly_layoff_event_count，使用 target_next_week_layoff_event_count 近似作为 TLCC 目标变量。"

# 重点新闻变量；如果原始变量不存在，尽量找相近列。
tlcc_priority = [
    "weekly_layoff_news_count",
    "weekly_layoff_news_share",
    "weekly_negative_share",
    "sentiment_shock",
    "weekly_avg_sentiment",
]

def find_close_variable(name, candidates):
    """在已有列中寻找与目标名称最接近的变量。"""
    if name in candidates:
        return name
    low_name = name.lower()
    tokens = [t for t in low_name.split("_") if t]
    scored = []
    for c in candidates:
        low = c.lower()
        score = sum(t in low for t in tokens)
        if score >= max(1, len(tokens) - 1):
            scored.append((score, c))
    if scored:
        return sorted(scored, reverse=True)[0][1]
    return None

tlcc_vars = []
for v in tlcc_priority:
    found = find_close_variable(v, df.columns)
    if found is not None and found not in tlcc_vars and not is_forbidden_feature(found):
        tlcc_vars.append(found)

# 如果重点变量一个都找不到，则退而使用已识别的新闻变量前若干个。
if not tlcc_vars:
    tlcc_vars = news_features[:5]

rows = []
for var in tlcc_vars:
    x0 = pd.to_numeric(df[var], errors="coerce")
    y0 = pd.to_numeric(df[tlcc_target], errors="coerce")
    for lag in range(-4, 5):
        # lag=1: corr(news_{t-1}, layoff_t)，所以对新闻变量 shift(1)。
        x = x0.shift(lag)
        pair = pd.DataFrame({"x": x, "y": y0}).dropna()
        if len(pair) >= 3 and pair["x"].std() > 0 and pair["y"].std() > 0:
            corr, pval = stats.pearsonr(pair["x"], pair["y"])
        else:
            corr, pval = np.nan, np.nan
        rows.append({
            "variable": var,
            "lag": lag,
            "correlation": corr,
            "n_obs": len(pair),
            "p_value": pval,
            "significance": sig_star(pval),
        })

tlcc_results = pd.DataFrame(rows)
tlcc_results.to_csv(RESULT_DIR / "tlcc_results.csv", index=False, encoding="utf-8-sig")

# 绘制 TLCC 折线图。
plt.figure(figsize=(10, 6))
for var in tlcc_vars:
    sub = tlcc_results[tlcc_results["variable"] == var]
    plt.plot(sub["lag"], sub["correlation"], marker="o", label=var)
plt.axhline(0, color="black", linewidth=0.8)
plt.axvline(0, color="gray", linestyle="--", linewidth=0.8)
plt.xlabel("Lag: positive means news leads layoffs")
plt.ylabel("Pearson correlation")
plt.title("TLCC between news variables and layoff events")
plt.legend(fontsize=8)
plot_save_close(FIG_DIR / "fig3_tlcc_lag_effect.png")

# 自动解释：重点看 lag1 和 lag2。
lead_sub = tlcc_results[tlcc_results["lag"].isin([1, 2])].copy()
lead_sub["abs_corr"] = lead_sub["correlation"].abs()
if len(lead_sub.dropna(subset=["correlation"])) > 0:
    best_lead = lead_sub.sort_values("abs_corr", ascending=False).iloc[0]
    best_sentence = (
        f"lag{int(best_lead['lag'])} 上相关最高的变量是 `{best_lead['variable']}`，"
        f"相关系数为 {best_lead['correlation']:.3f}，p 值为 {best_lead['p_value']:.3f}{best_lead['significance']}。"
    )
    if pd.notna(best_lead["p_value"]) and best_lead["p_value"] < 0.10:
        lead_conclusion = "这表明新闻变量与之后裁员事件存在一定滞后关联，具有前瞻信息迹象。"
    else:
        lead_conclusion = "但该相关性未达到常用显著性水平，因此只能作为探索性证据，不能夸大为稳定领先关系。"
else:
    best_sentence = "lag1/lag2 没有足够有效观测用于判断最高相关变量。"
    lead_conclusion = "当前数据不足以支持新闻领先裁员的 TLCC 判断。"

interpretation = f"""# TLCC 时滞交叉相关解释

{tlcc_target_note}

本部分计算新闻变量在 `t-k` 周与裁员事件数在 `t` 周之间的相关系数。`lag=1` 表示新闻变量领先裁员事件 1 周，`lag=2` 表示领先 2 周，`lag<0` 表示新闻变量滞后于裁员事件。

- 使用的新闻变量：{describe_available(tlcc_vars)}
- {best_sentence}
- {lead_conclusion}

需要注意，TLCC 是双变量相关分析，未控制宏观变量和历史裁员状态，因此只能说明滞后关联方向，不能证明因果关系。
"""
safe_write_text(TEXT_DIR / "tlcc_interpretation.md", interpretation)

display(tlcc_results)
print(interpretation)


                    variable  lag  correlation  n_obs       p_value significance
0   weekly_layoff_news_count   -4     0.037061    219  5.854056e-01             
1   weekly_layoff_news_count   -3     0.172046    220  1.057611e-02           **
2   weekly_layoff_news_count   -2     0.129731    221  5.412892e-02            *
3   weekly_layoff_news_count   -1     0.230556    222  5.350126e-04          ***
4   weekly_layoff_news_count    0     0.330269    223  4.498029e-07          ***
5   weekly_layoff_news_count    1     0.242761    222  2.607964e-04          ***
6   weekly_layoff_news_count    2     0.225294    221  7.415834e-04          ***
7   weekly_layoff_news_count    3     0.095820    220  1.566573e-01             
8   weekly_layoff_news_count    4     0.117611    219  8.246601e-02            *
9   weekly_layoff_news_share   -4     0.103252    217  1.294525e-01             
10  weekly_layoff_news_share   -3     0.161046    218  1.732663e-02           **
11  weekly_layoff_news_share

## 第 5 部分：RQ1 机制检验二：HAC 稳健回归 / Poisson / Negative Binomial

本部分用回归检验新闻滞后变量是否能解释下一周裁员事件数。OLS 模型使用 Newey-West / HAC 稳健标准误；计数模型使用 Poisson GLM，并在可收敛时尝试 Negative Binomial GLM。


In [7]:
# ==============================
# 第 5 部分：RQ1 HAC / Poisson / Negative Binomial 回归
# ==============================

regression_rows = []
model_fit_rows = []

# 因变量：优先 log 下一周事件数。
y_log_col = main_log_target if main_log_target in df.columns else None
if y_log_col is None:
    df[main_log_target] = np.log1p(df[main_target])
    y_log_col = main_log_target

# 核心新闻 lag1 / lag2 变量。
core_news_lag1 = [
    "weekly_layoff_news_count_lag1",
    "weekly_layoff_news_share_lag1",
    "weekly_negative_share_lag1",
    "sentiment_shock_lag1",
]
core_news_lag2 = [
    "weekly_layoff_news_count_lag2",
    "weekly_layoff_news_share_lag2",
    "weekly_negative_share_lag2",
    "sentiment_shock_lag2",
]
# 如果 lag 列不存在，但原始变量存在，则这里不强行替代；滞后回归需要明确 lag。
core_news_lag1 = numeric_existing_features(df, core_news_lag1)
core_news_lag2 = numeric_existing_features(df, core_news_lag2)

ar_controls = numeric_existing_features(df, ["weekly_layoff_event_count_lag1", "weekly_layoff_event_count_lag2"])
if not ar_controls and "weekly_layoff_event_count" in df.columns:
    ar_controls = ["weekly_layoff_event_count"]

macro_controls = macro_features[:8]  # 控制变量数量过多时容易在小样本中不稳定，因此限制前若干个。
extra_controls = numeric_existing_features(df, ["has_news_week"])

reg_specs = {
    "R1_AR_only": ar_controls,
    "R2_AR_news_lag1": ar_controls + core_news_lag1 + extra_controls,
    "R3_AR_news_lag12_macro": ar_controls + core_news_lag1 + core_news_lag2 + macro_controls + extra_controls,
}

# OLS + HAC。
for model_name, features in reg_specs.items():
    features = numeric_existing_features(df, features)
    if not features:
        model_fit_rows.append({"model": model_name, "status": "skipped_no_features"})
        continue
    reg_df = df[[y_log_col] + features].copy()
    for c in [y_log_col] + features:
        reg_df[c] = pd.to_numeric(reg_df[c], errors="coerce")
    reg_df = reg_df.dropna()
    if len(reg_df) < max(10, len(features) + 5):
        model_fit_rows.append({"model": model_name, "status": f"skipped_small_sample_n={len(reg_df)}"})
        continue
    X = sm.add_constant(reg_df[features], has_constant="add")
    y = reg_df[y_log_col]
    try:
        fit = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
        model_fit_rows.append({
            "model": model_name,
            "status": "ok",
            "n_obs": int(fit.nobs),
            "r_squared": fit.rsquared,
            "adj_r_squared": fit.rsquared_adj,
            "aic": fit.aic,
            "bic": fit.bic,
        })
        for term in fit.params.index:
            regression_rows.append({
                "model": model_name,
                "model_type": "OLS_HAC",
                "term": term,
                "coef": fit.params.get(term, np.nan),
                "std_err": fit.bse.get(term, np.nan),
                "stat": fit.tvalues.get(term, np.nan),
                "p_value": fit.pvalues.get(term, np.nan),
                "significance": sig_star(fit.pvalues.get(term, np.nan)),
                "n_obs": int(fit.nobs),
                "r_squared": fit.rsquared,
                "adj_r_squared": fit.rsquared_adj,
            })
    except Exception as e:
        model_fit_rows.append({"model": model_name, "status": "failed", "error": str(e)})

# Poisson GLM：因变量使用事件数。
r3_features = numeric_existing_features(df, reg_specs.get("R3_AR_news_lag12_macro", []))
if r3_features:
    pois_df = df[[main_target] + r3_features].copy()
    for c in [main_target] + r3_features:
        pois_df[c] = pd.to_numeric(pois_df[c], errors="coerce")
    pois_df = pois_df.dropna()
    if len(pois_df) >= max(10, len(r3_features) + 5):
        Xp = sm.add_constant(pois_df[r3_features], has_constant="add")
        yp = np.clip(pois_df[main_target], 0, None)
        try:
            pois_fit = sm.GLM(yp, Xp, family=sm.families.Poisson()).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
            model_fit_rows.append({"model": "R4_Poisson_R3_features", "status": "ok", "n_obs": int(pois_fit.nobs), "aic": pois_fit.aic, "bic": getattr(pois_fit, "bic", np.nan)})
            for term in pois_fit.params.index:
                regression_rows.append({
                    "model": "R4_Poisson_R3_features",
                    "model_type": "Poisson_GLM_HAC",
                    "term": term,
                    "coef": pois_fit.params.get(term, np.nan),
                    "std_err": pois_fit.bse.get(term, np.nan),
                    "stat": pois_fit.tvalues.get(term, np.nan),
                    "p_value": pois_fit.pvalues.get(term, np.nan),
                    "significance": sig_star(pois_fit.pvalues.get(term, np.nan)),
                    "n_obs": int(pois_fit.nobs),
                    "r_squared": np.nan,
                    "adj_r_squared": np.nan,
                })
        except Exception as e:
            model_fit_rows.append({"model": "R4_Poisson_R3_features", "status": "failed", "error": str(e)})
        try:
            nb_fit = sm.GLM(yp, Xp, family=sm.families.NegativeBinomial()).fit(maxiter=200)
            model_fit_rows.append({"model": "R5_NegativeBinomial_R3_features", "status": "ok", "n_obs": int(nb_fit.nobs), "aic": nb_fit.aic, "bic": getattr(nb_fit, "bic", np.nan)})
            for term in nb_fit.params.index:
                regression_rows.append({
                    "model": "R5_NegativeBinomial_R3_features",
                    "model_type": "NegativeBinomial_GLM",
                    "term": term,
                    "coef": nb_fit.params.get(term, np.nan),
                    "std_err": nb_fit.bse.get(term, np.nan),
                    "stat": nb_fit.tvalues.get(term, np.nan),
                    "p_value": nb_fit.pvalues.get(term, np.nan),
                    "significance": sig_star(nb_fit.pvalues.get(term, np.nan)),
                    "n_obs": int(nb_fit.nobs),
                    "r_squared": np.nan,
                    "adj_r_squared": np.nan,
                })
        except Exception as e:
            model_fit_rows.append({"model": "R5_NegativeBinomial_R3_features", "status": "failed_or_not_converged", "error": str(e)})
    else:
        model_fit_rows.append({"model": "R4_R5_count_models", "status": f"skipped_small_sample_n={len(pois_df)}"})
else:
    model_fit_rows.append({"model": "R4_R5_count_models", "status": "skipped_no_r3_features"})

regression_results = pd.DataFrame(regression_rows)
regression_fits = pd.DataFrame(model_fit_rows)
regression_results.to_csv(RESULT_DIR / "regression_results.csv", index=False, encoding="utf-8-sig")
regression_fits.to_csv(RESULT_DIR / "regression_model_fit_summary.csv", index=False, encoding="utf-8-sig")

# 自动解释核心变量：重点看 weekly_layoff_news_count_lag1。
key = "weekly_layoff_news_count_lag1"
key_rows = regression_results[regression_results["term"] == key].copy() if not regression_results.empty else pd.DataFrame()
if not key_rows.empty:
    key_desc = []
    for _, r in key_rows.iterrows():
        direction = "正" if r["coef"] > 0 else "负"
        sig_text = "显著" if pd.notna(r["p_value"]) and r["p_value"] < 0.10 else "不显著"
        key_desc.append(f"- {r['model']} 中 `{key}` 系数为{direction}（coef={r['coef']:.4f}, p={r['p_value']:.3f}{r['significance']}），{sig_text}。")
    key_text = "\n".join(key_desc)
else:
    key_text = f"- 未找到 `{key}`，可能是成员A矩阵中没有生成该滞后变量，因此无法直接检验它。"

r2_fit = regression_fits[regression_fits["model"] == "R2_AR_news_lag1"]
r1_fit = regression_fits[regression_fits["model"] == "R1_AR_only"]
r3_fit = regression_fits[regression_fits["model"] == "R3_AR_news_lag12_macro"]
fit_text = ""
if not r1_fit.empty and not r2_fit.empty:
    try:
        fit_text += f"R1 adjusted R²={float(r1_fit.iloc[0].get('adj_r_squared', np.nan)):.3f}；R2 adjusted R²={float(r2_fit.iloc[0].get('adj_r_squared', np.nan)):.3f}。\n"
    except Exception:
        pass
if not r3_fit.empty:
    try:
        fit_text += f"R3 adjusted R²={float(r3_fit.iloc[0].get('adj_r_squared', np.nan)):.3f}。\n"
    except Exception:
        pass

reg_interpretation = f"""# 滞后回归解释

本部分以 `{y_log_col}` 作为 OLS 因变量，并使用 Newey-West / HAC 稳健标准误（maxlags=4）。Poisson GLM 使用 `{main_target}` 作为计数因变量；Negative Binomial 在可收敛时运行。

## 核心新闻变量解释

{key_text}

## 模型解释力变化

{fit_text if fit_text else '由于可用变量或样本限制，部分模型未能给出可比较的 adjusted R²。'}

## 谨慎解释

如果新闻变量系数为正且在 10% 或 5% 水平显著，可以解释为新闻裁员强度与下一周裁员事件数存在滞后关联。如果结果不显著，则可能来自周度样本量有限、新闻文本噪声、科技裁员事件突发性、以及宏观控制变量与新闻变量之间的共线性。上述结果不构成因果证明，只支持“前瞻信息”或“滞后关联”的讨论。
"""
safe_write_text(TEXT_DIR / "regression_interpretation.md", reg_interpretation)

display(regression_results)
display(regression_fits)
print(reg_interpretation)


                              model            model_type                            term       coef    std_err       stat       p_value significance  n_obs  r_squared  adj_r_squared
0                        R1_AR_only               OLS_HAC                           const   1.686513   0.118122  14.277686  3.014443e-46          ***    221   0.000552      -0.008617
1                        R1_AR_only               OLS_HAC  weekly_layoff_event_count_lag1   0.000390   0.007664   0.050871  9.594285e-01                 221   0.000552      -0.008617
2                        R1_AR_only               OLS_HAC  weekly_layoff_event_count_lag2  -0.002462   0.006501  -0.378705  7.049067e-01                 221   0.000552      -0.008617
3                   R2_AR_news_lag1               OLS_HAC                           const   2.890491   0.551625   5.239959  1.606125e-07          ***    218   0.045384       0.013563
4                   R2_AR_news_lag1               OLS_HAC  weekly_layoff_event_count_

## 第 6 部分：RQ2 预测模型一：Macro Only / News Only / Macro + News

本部分比较四组特征集的样本外预测表现：AR Baseline、Macro Only、News Only、Macro + News。所有模型使用相同的时间测试集，线性模型和 Poisson 使用标准化 Pipeline，树模型不强制标准化。


In [8]:
# ==============================
# 第 6 部分：RQ2 预测模型比较
# ==============================

# 四组特征集。
feature_sets = {
    "AR Baseline": ar_features,
    "Macro Only": ar_features + macro_features,
    "News Only": ar_features + news_features,
    "Macro + News": ar_features + macro_features + news_features,
}
feature_sets = {k: numeric_existing_features(df, v) for k, v in feature_sets.items()}

# TimeSeriesSplit 的折数根据训练样本自动调整，避免小样本报错。
n_train = len(train_df)
n_splits = min(5, max(2, n_train // 12)) if n_train >= 24 else 2
tscv = TimeSeriesSplit(n_splits=n_splits)

# 模型构造函数。Ridge/Lasso 使用 log1p 目标训练，再 expm1 转回事件数。
def build_models(n_train):
    models = {}
    # alpha 网格较保守，适合小样本。
    alphas = np.logspace(-3, 3, 25)
    models["Ridge"] = {
        "estimator": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", RidgeCV(alphas=alphas, cv=tscv)),
        ]),
        "target_transform": "log1p",
    }
    models["Lasso"] = {
        "estimator": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LassoCV(alphas=alphas, cv=tscv, random_state=RANDOM_STATE, max_iter=20000)),
        ]),
        "target_transform": "log1p",
    }
    models["PoissonRegressor"] = {
        "estimator": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", PoissonRegressor(alpha=0.1, max_iter=2000)),
        ]),
        "target_transform": "raw_nonnegative",
    }
    models["RandomForest"] = {
        "estimator": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(n_estimators=500, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
        "target_transform": "log1p",
    }
    if XGBOOST_AVAILABLE:
        models["XGBoost"] = {
            "estimator": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", XGBRegressor(
                    n_estimators=300,
                    max_depth=3,
                    learning_rate=0.05,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    objective="reg:squarederror",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )),
            ]),
            "target_transform": "log1p",
        }
    return models

models = build_models(n_train)

baseline_rows = []
prediction_records = []

for set_name, feats in feature_sets.items():
    feats = numeric_existing_features(df, feats)
    if not feats:
        baseline_rows.append({"feature_set": set_name, "model": "ALL", "status": "skipped_no_features"})
        continue
    X_train, y_train_raw, used_feats, train_mask_y = prepare_X_y(train_df, feats, main_target)
    X_test, y_test_raw, used_feats_test, test_mask_y = prepare_X_y(test_df, feats, main_target)
    # 对齐训练、测试的特征列。
    used_feats = clean_feature_list(used_feats, X_test.columns)
    X_train = X_train[used_feats]
    X_test = X_test[used_feats]
    if len(X_train) < 10 or len(X_test) < 2:
        baseline_rows.append({"feature_set": set_name, "model": "ALL", "status": f"skipped_small_sample_train={len(X_train)}_test={len(X_test)}"})
        continue
    for model_name, spec in models.items():
        estimator = spec["estimator"]
        transform = spec["target_transform"]
        try:
            if transform == "log1p":
                y_train = np.log1p(np.clip(y_train_raw, 0, None))
                estimator.fit(X_train, y_train)
                pred = np.expm1(estimator.predict(X_test))
            else:
                y_train = np.clip(y_train_raw, 0, None)
                estimator.fit(X_train, y_train)
                pred = estimator.predict(X_test)
            pred = np.clip(pred, 0, None)
            metrics = evaluate_predictions(y_test_raw, pred)
            row = {"feature_set": set_name, "model": model_name, "status": "ok", "n_features": len(used_feats), "n_train": len(X_train), "n_test": len(X_test)}
            row.update(metrics)
            baseline_rows.append(row)
            for t, yt, yp in zip(test_df.loc[test_mask_y, time_col], y_test_raw, pred):
                prediction_records.append({
                    "feature_set": set_name,
                    "model": model_name,
                    "time": t,
                    "actual": yt,
                    "predicted": yp,
                })
        except Exception as e:
            baseline_rows.append({"feature_set": set_name, "model": model_name, "status": "failed", "error": str(e), "n_features": len(used_feats)})

baseline_model_results = pd.DataFrame(baseline_rows)
baseline_model_results.to_csv(RESULT_DIR / "baseline_model_results.csv", index=False, encoding="utf-8-sig")

# model_comparison 聚合保留 ok 行，并按 MAE 排序。
model_comparison = baseline_model_results[baseline_model_results["status"] == "ok"].copy()
if not model_comparison.empty:
    model_comparison = model_comparison.sort_values(["MAE", "RMSE"]).reset_index(drop=True)
model_comparison.to_csv(RESULT_DIR / "model_comparison.csv", index=False, encoding="utf-8-sig")

prediction_df = pd.DataFrame(prediction_records)
prediction_df.to_csv(RESULT_DIR / "prediction_records.csv", index=False, encoding="utf-8-sig")

# 图 1：不同特征集/模型的 MAE 与 RMSE。
if not model_comparison.empty:
    plot_df = model_comparison.copy()
    plot_df["label"] = plot_df["feature_set"] + "\n" + plot_df["model"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].barh(plot_df["label"], plot_df["MAE"])
    axes[0].set_title("MAE by model")
    axes[0].set_xlabel("MAE")
    axes[1].barh(plot_df["label"], plot_df["RMSE"])
    axes[1].set_title("RMSE by model")
    axes[1].set_xlabel("RMSE")
    plot_save_close(FIG_DIR / "fig4_model_comparison_mae_rmse.png")

    # 图 2：最佳模型的真实值 vs 预测值。
    best = model_comparison.iloc[0]
    best_pred = prediction_df[(prediction_df["feature_set"] == best["feature_set"]) & (prediction_df["model"] == best["model"])].copy()
    plt.figure(figsize=(10, 5))
    plt.plot(pd.to_datetime(best_pred["time"]), best_pred["actual"], marker="o", label="Actual")
    plt.plot(pd.to_datetime(best_pred["time"]), best_pred["predicted"], marker="o", label="Predicted")
    plt.title(f"Prediction vs actual: {best['feature_set']} / {best['model']}")
    plt.xlabel("Time")
    plt.ylabel("Next-week layoff event count")
    plt.legend()
    plot_save_close(FIG_DIR / "prediction_vs_actual_best_model.png")

# 自动解释 Macro + News 是否优于 Macro Only。
if not model_comparison.empty:
    comparison_lines = []
    for model_name in sorted(model_comparison["model"].unique()):
        sub = model_comparison[model_comparison["model"] == model_name]
        macro = sub[sub["feature_set"] == "Macro Only"]
        macro_news = sub[sub["feature_set"] == "Macro + News"]
        if not macro.empty and not macro_news.empty:
            m0 = macro.iloc[0]
            m1 = macro_news.iloc[0]
            mae_change = m0["MAE"] - m1["MAE"]
            rmse_change = m0["RMSE"] - m1["RMSE"]
            comparison_lines.append(
                f"- {model_name}: Macro + News 相比 Macro Only 的 MAE 变化为 {mae_change:.3f}，RMSE 变化为 {rmse_change:.3f}。"
            )
    if comparison_lines:
        comparison_text = "\n".join(comparison_lines)
    else:
        comparison_text = "缺少可直接比较的 Macro Only 与 Macro + News 结果。"
    best_text = f"样本外 MAE 最低的组合是 `{model_comparison.iloc[0]['feature_set']}` / `{model_comparison.iloc[0]['model']}`，MAE={model_comparison.iloc[0]['MAE']:.3f}，RMSE={model_comparison.iloc[0]['RMSE']:.3f}。"
else:
    comparison_text = "由于可用特征或样本不足，预测模型未能成功运行。"
    best_text = "没有可用最佳模型。"

model_interp = f"""# 模型比较解释

{best_text}

## Macro Only vs Macro + News

{comparison_text}

如果 Macro + News 在 MAE 或 RMSE 上下降，可以解释为新闻变量提供了样本外预测增量；如果下降很小或没有下降，应更谨慎地说明新闻变量可能主要体现机制解释价值，而非稳定预测优势。
"""
safe_write_text(TEXT_DIR / "model_comparison_interpretation.md", model_interp)

display(baseline_model_results)
display(model_comparison)
print(model_interp)


     feature_set             model status  n_features  n_train  n_test       MAE       RMSE  MAPE_nonzero     sMAPE  Poisson_Deviance  Directional_Accuracy
0    AR Baseline             Ridge     ok           5      157      66  2.467472   2.987546      0.662283  0.925723          2.872423              0.661538
1    AR Baseline             Lasso     ok           5      157      66  2.555816   3.050798      0.694106  0.923955          2.957580              0.661538
2    AR Baseline  PoissonRegressor     ok           5      157      66  4.463018   4.966139      1.317908  1.016324          5.619265              0.661538
3    AR Baseline      RandomForest     ok           5      157      66  2.472838   3.284904      0.696380  1.092653          3.752133              0.646154
4     Macro Only             Ridge     ok          17      157      66  2.449189   3.183926      0.645480  1.124776          4.101109              0.646154
5     Macro Only             Lasso     ok          17      157  

## 第 7 部分：RQ2 消融实验

本部分聚焦比较 Macro Only 与 Macro + News。核心指标是加入新闻变量后 MAE/RMSE 的百分比改善幅度。


In [9]:
# ==============================
# 第 7 部分：RQ2 消融实验
# ==============================

ablation_rows = []
if 'model_comparison' in globals() and not model_comparison.empty:
    for model_name in sorted(model_comparison["model"].unique()):
        sub = model_comparison[model_comparison["model"] == model_name]
        macro = sub[sub["feature_set"] == "Macro Only"]
        macro_news = sub[sub["feature_set"] == "Macro + News"]
        if not macro.empty and not macro_news.empty:
            m0 = macro.iloc[0]
            m1 = macro_news.iloc[0]
            improvement_mae_pct = (m0["MAE"] - m1["MAE"]) / m0["MAE"] * 100 if m0["MAE"] != 0 else np.nan
            improvement_rmse_pct = (m0["RMSE"] - m1["RMSE"]) / m0["RMSE"] * 100 if m0["RMSE"] != 0 else np.nan
            ablation_rows.append({
                "model": model_name,
                "MAE_macro_only": m0["MAE"],
                "MAE_macro_news": m1["MAE"],
                "RMSE_macro_only": m0["RMSE"],
                "RMSE_macro_news": m1["RMSE"],
                "improvement_mae_pct": improvement_mae_pct,
                "improvement_rmse_pct": improvement_rmse_pct,
            })

ablation_table = pd.DataFrame(ablation_rows)
ablation_table.to_csv(RESULT_DIR / "ablation_table.csv", index=False, encoding="utf-8-sig")

if not ablation_table.empty:
    x = np.arange(len(ablation_table))
    width = 0.35
    plt.figure(figsize=(10, 5))
    plt.bar(x - width/2, ablation_table["improvement_mae_pct"], width, label="MAE improvement %")
    plt.bar(x + width/2, ablation_table["improvement_rmse_pct"], width, label="RMSE improvement %")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.xticks(x, ablation_table["model"], rotation=30, ha="right")
    plt.ylabel("Improvement (%)")
    plt.title("Ablation: Macro Only vs Macro + News")
    plt.legend()
    plot_save_close(FIG_DIR / "ablation_macro_vs_macro_news.png")

    positive_mae = (ablation_table["improvement_mae_pct"] > 0).sum()
    positive_rmse = (ablation_table["improvement_rmse_pct"] > 0).sum()
    ablation_text = f"在 {len(ablation_table)} 个可比较模型中，Macro + News 有 {positive_mae} 个模型降低 MAE，有 {positive_rmse} 个模型降低 RMSE。"
    if positive_mae > 0 or positive_rmse > 0:
        ablation_conclusion = "这说明新闻变量在部分模型中提供了样本外预测增量，但仍需结合提升幅度判断其经济意义。"
    else:
        ablation_conclusion = "当前样本下新闻变量没有稳定改善预测表现，更适合被解释为机制关联变量，而非稳定预测增强变量。"
else:
    ablation_text = "缺少 Macro Only 与 Macro + News 的成对模型结果，无法计算消融实验。"
    ablation_conclusion = "需要更多有效特征或样本后再判断新闻变量的增量预测价值。"

ablation_interp = f"""# 消融实验解释

{ablation_text}

{ablation_conclusion}

计算公式：`improvement_mae_pct = (MAE_macro_only - MAE_macro_news) / MAE_macro_only * 100`；RMSE 同理。正值表示加入新闻变量后误差下降。
"""
safe_write_text(TEXT_DIR / "ablation_interpretation.md", ablation_interp)

display(ablation_table)
print(ablation_interp)


              model  MAE_macro_only  MAE_macro_news  RMSE_macro_only  RMSE_macro_news  improvement_mae_pct  improvement_rmse_pct
0             Lasso        2.341831        3.599813         2.830692         4.709694           -53.717857            -66.379583
1  PoissonRegressor        2.547090        3.111649         3.229614         4.319356           -22.164872            -33.742209
2      RandomForest        2.294883        3.701607         3.005709         4.458741           -61.298300            -48.342395
3             Ridge        2.449189        3.943169         3.183926         5.778139           -60.998984            -81.478470
# 消融实验解释

在 4 个可比较模型中，Macro + News 有 0 个模型降低 MAE，有 0 个模型降低 RMSE。

当前样本下新闻变量没有稳定改善预测表现，更适合被解释为机制关联变量，而非稳定预测增强变量。

计算公式：`improvement_mae_pct = (MAE_macro_only - MAE_macro_news) / MAE_macro_only * 100`；RMSE 同理。正值表示加入新闻变量后误差下降。



## 第 8 部分：机器学习模型与特征重要性

本部分针对 Macro + News 特征集训练 Random Forest，并在 XGBoost 可用时训练 XGBoost，输出前 20 个重要特征。


In [10]:
# ==============================
# 第 8 部分：机器学习模型与特征重要性
# ==============================

ml_rows = []
importance_outputs = {}
macro_news_features = numeric_existing_features(df, feature_sets.get("Macro + News", []))

if macro_news_features:
    X_train, y_train_raw, used_feats, train_mask_y = prepare_X_y(train_df, macro_news_features, main_target)
    X_test, y_test_raw, used_feats_test, test_mask_y = prepare_X_y(test_df, macro_news_features, main_target)
    used_feats = clean_feature_list(used_feats, X_test.columns)
    X_train = X_train[used_feats]
    X_test = X_test[used_feats]

    if len(X_train) >= 10 and len(X_test) >= 2:
        # Random Forest 特征重要性。
        rf_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(n_estimators=800, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)),
        ])
        rf_pipe.fit(X_train, np.log1p(np.clip(y_train_raw, 0, None)))
        rf_pred = np.expm1(rf_pipe.predict(X_test))
        rf_metrics = evaluate_predictions(y_test_raw, rf_pred)
        ml_rows.append({"model": "RandomForest", "status": "ok", "n_features": len(used_feats), **rf_metrics})
        rf_model = rf_pipe.named_steps["model"]
        rf_imp = pd.DataFrame({"feature": used_feats, "importance": rf_model.feature_importances_}).sort_values("importance", ascending=False).head(20)
        rf_imp.to_csv(RESULT_DIR / "feature_importance_random_forest.csv", index=False, encoding="utf-8-sig")
        importance_outputs["RandomForest"] = rf_imp
        plt.figure(figsize=(8, 6))
        plt.barh(rf_imp.sort_values("importance")["feature"], rf_imp.sort_values("importance")["importance"])
        plt.title("Random Forest feature importance")
        plt.xlabel("Importance")
        plot_save_close(FIG_DIR / "feature_importance_random_forest.png")

        # XGBoost 特征重要性。
        if XGBOOST_AVAILABLE:
            try:
                xgb_pipe = Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("model", XGBRegressor(
                        n_estimators=400,
                        max_depth=3,
                        learning_rate=0.05,
                        subsample=0.8,
                        colsample_bytree=0.8,
                        objective="reg:squarederror",
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    )),
                ])
                xgb_pipe.fit(X_train, np.log1p(np.clip(y_train_raw, 0, None)))
                xgb_pred = np.expm1(xgb_pipe.predict(X_test))
                xgb_metrics = evaluate_predictions(y_test_raw, xgb_pred)
                ml_rows.append({"model": "XGBoost", "status": "ok", "n_features": len(used_feats), **xgb_metrics})
                xgb_model = xgb_pipe.named_steps["model"]
                xgb_imp = pd.DataFrame({"feature": used_feats, "importance": xgb_model.feature_importances_}).sort_values("importance", ascending=False).head(20)
                xgb_imp.to_csv(RESULT_DIR / "feature_importance_xgboost.csv", index=False, encoding="utf-8-sig")
                importance_outputs["XGBoost"] = xgb_imp
                plt.figure(figsize=(8, 6))
                plt.barh(xgb_imp.sort_values("importance")["feature"], xgb_imp.sort_values("importance")["importance"])
                plt.title("XGBoost feature importance")
                plt.xlabel("Importance")
                plot_save_close(FIG_DIR / "feature_importance_xgboost.png")
            except Exception as e:
                ml_rows.append({"model": "XGBoost", "status": "failed", "error": str(e)})
        else:
            ml_rows.append({"model": "XGBoost", "status": "skipped_not_installed", "error": globals().get("XGBOOST_IMPORT_ERROR", "not installed")})
    else:
        ml_rows.append({"model": "ML_importance", "status": f"skipped_small_sample_train={len(X_train)}_test={len(X_test)}"})
else:
    ml_rows.append({"model": "ML_importance", "status": "skipped_no_macro_news_features"})

ml_model_results = pd.DataFrame(ml_rows)
ml_model_results.to_csv(RESULT_DIR / "ml_model_results.csv", index=False, encoding="utf-8-sig")

# 如果因为无特征或小样本未生成 RF importance，也创建空文件以便输出清单完整。
if not (RESULT_DIR / "feature_importance_random_forest.csv").exists():
    pd.DataFrame(columns=["feature", "importance"]).to_csv(RESULT_DIR / "feature_importance_random_forest.csv", index=False, encoding="utf-8-sig")

# 自动解释特征重要性。
importance_lines = []
for model_name, imp in importance_outputs.items():
    top_features = imp["feature"].tolist()
    news_in_top = [f for f in top_features if f in news_features]
    ar_in_top = [f for f in top_features if f in ar_features]
    macro_in_top = [f for f in top_features if f in macro_features]
    importance_lines.append(
        f"- {model_name}: 前 20 个重要特征中，新闻变量 {len(news_in_top)} 个（{describe_available(news_in_top)}），"
        f"AR 变量 {len(ar_in_top)} 个（{describe_available(ar_in_top)}），宏观变量 {len(macro_in_top)} 个（{describe_available(macro_in_top)}）。"
    )
if not importance_lines:
    importance_lines.append("- 由于可用特征、样本量或模型依赖限制，未能生成有效特征重要性。")

importance_interp = "# 机器学习特征重要性解释\n\n" + "\n".join(importance_lines) + "\n\n如果滞后裁员变量主导，说明下一周事件数主要受近期裁员状态影响；如果新闻变量进入前 20，则说明新闻情绪或裁员新闻强度在非线性模型中具有一定预测信息。"
safe_write_text(TEXT_DIR / "feature_importance_interpretation.md", importance_interp)

display(ml_model_results)
for name, imp in importance_outputs.items():
    print(name)
    display(imp)
print(importance_interp)


          model                 status  n_features       MAE      RMSE  MAPE_nonzero     sMAPE  Poisson_Deviance  Directional_Accuracy                      error
0  RandomForest                     ok        70.0  3.581728  4.296737      0.865991  0.968175          4.708956              0.569231                        NaN
1       XGBoost  skipped_not_installed         NaN       NaN       NaN           NaN       NaN               NaN                   NaN  No module named 'xgboost'
RandomForest
                                 feature  importance
64    weekly_hiring_news_count_roll4_sum    0.094423
3     log_weekly_layoff_event_count_lag1    0.053019
0         weekly_layoff_event_count_lag1    0.049786
2   weekly_layoff_event_count_roll4_mean    0.046686
1         weekly_layoff_event_count_lag2    0.044728
4     log_weekly_layoff_event_count_lag2    0.043850
61         weekly_hiring_news_count_lag4    0.038366
58         weekly_hiring_news_count_lag1    0.030406
54            weekly_neg

## 第 9 部分：稳健性检验

本部分进行三类稳健性检验：更换目标变量、使用更保守的 layoff-only 新闻变量集合、调整新闻滞后窗口。目标是检查主要结论是否依赖某一种标签或特征设定。


In [11]:
# ==============================
# 第 9 部分：稳健性检验
# ==============================

robust_rows = []

def run_simple_compare(target_col, feature_set_a, feature_set_b, label, models_to_run=("Ridge", "RandomForest")):
    """用于稳健性检验的简化 Macro Only vs Macro + News 对比。"""
    local_rows = []
    if target_col not in df.columns:
        return [{"robustness_check": label, "status": f"skipped_missing_target_{target_col}"}]
    for set_name, feats in [("Macro Only", feature_set_a), ("Macro + News", feature_set_b)]:
        feats = numeric_existing_features(df, feats)
        if not feats:
            local_rows.append({"robustness_check": label, "feature_set": set_name, "status": "skipped_no_features"})
            continue
        X_train, y_train_raw, used_feats, _ = prepare_X_y(train_df, feats, target_col)
        X_test, y_test_raw, _, _ = prepare_X_y(test_df, feats, target_col)
        used_feats = clean_feature_list(used_feats, X_test.columns)
        X_train = X_train[used_feats]
        X_test = X_test[used_feats]
        if len(X_train) < 10 or len(X_test) < 2:
            local_rows.append({"robustness_check": label, "feature_set": set_name, "status": "skipped_small_sample"})
            continue
        for model_name in models_to_run:
            try:
                if model_name == "Ridge":
                    est = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))])
                elif model_name == "RandomForest":
                    est = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1))])
                else:
                    continue
                y_train = np.log1p(np.clip(y_train_raw, 0, None))
                est.fit(X_train, y_train)
                pred = np.expm1(est.predict(X_test))
                metrics = evaluate_predictions(y_test_raw, pred)
                local_rows.append({"robustness_check": label, "target": target_col, "feature_set": set_name, "model": model_name, "status": "ok", **metrics})
            except Exception as e:
                local_rows.append({"robustness_check": label, "feature_set": set_name, "model": model_name, "status": "failed", "error": str(e)})
    return local_rows

# 稳健性 1：更换目标变量。
alt_targets = ["target_next_week_known_layoff_count", "target_next_week_log_known_layoff_count"]
alt_target = next((t for t in alt_targets if t in df.columns), None)
if alt_target is not None:
    # 如果目标是 log 版本，评估时仍作为数值目标处理；解释时说明为辅助目标。
    robust_rows.extend(run_simple_compare(alt_target, ar_features + macro_features, ar_features + macro_features + news_features, "alternative_target_known_layoff"))
else:
    robust_rows.append({"robustness_check": "alternative_target_known_layoff", "status": "skipped_no_known_layoff_target"})

# 稳健性 2：更保守的 layoff-only news 特征集合。
layoff_only_base = ["weekly_layoff_news_count", "weekly_layoff_news_share", "weekly_layoff_avg_sentiment"]
layoff_only_news = []
for b in layoff_only_base:
    layoff_only_news.append(b)
    for k in range(1, 5):
        layoff_only_news.append(f"{b}_lag{k}")
layoff_only_news = numeric_existing_features(df, layoff_only_news)
if layoff_only_news:
    robust_rows.extend(run_simple_compare(main_target, ar_features + macro_features, ar_features + macro_features + layoff_only_news, "layoff_only_news_features"))
else:
    robust_rows.append({"robustness_check": "layoff_only_news_features", "status": "skipped_no_layoff_only_news_features"})

# 稳健性 3：滞后窗口更换。
for label, lag_list in [("news_lag1_only", [1]), ("news_lag1_lag2", [1, 2]), ("news_lag1_to_lag4", [1, 2, 3, 4])]:
    lag_feats = []
    for c in news_features:
        low = c.lower()
        if any(low.endswith(f"_lag{k}") for k in lag_list):
            lag_feats.append(c)
    lag_feats = numeric_existing_features(df, lag_feats)
    if lag_feats:
        robust_rows.extend(run_simple_compare(main_target, ar_features + macro_features, ar_features + macro_features + lag_feats, label))
    else:
        robust_rows.append({"robustness_check": label, "status": "skipped_no_matching_lag_features"})

robustness_results = pd.DataFrame(robust_rows)
robustness_results.to_csv(RESULT_DIR / "robustness_results.csv", index=False, encoding="utf-8-sig")

# 自动解释稳健性。
ok_robust = robustness_results[robustness_results.get("status", pd.Series(dtype=str)) == "ok"].copy() if not robustness_results.empty else pd.DataFrame()
robust_lines = []
if not ok_robust.empty:
    for check in ok_robust["robustness_check"].dropna().unique():
        sub = ok_robust[ok_robust["robustness_check"] == check]
        improved = 0
        total = 0
        for model_name in sub["model"].dropna().unique():
            m0 = sub[(sub["model"] == model_name) & (sub["feature_set"] == "Macro Only")]
            m1 = sub[(sub["model"] == model_name) & (sub["feature_set"] == "Macro + News")]
            if not m0.empty and not m1.empty:
                total += 1
                if m1.iloc[0]["MAE"] < m0.iloc[0]["MAE"] or m1.iloc[0]["RMSE"] < m0.iloc[0]["RMSE"]:
                    improved += 1
        robust_lines.append(f"- {check}: {total} 个可比较模型中，{improved} 个模型在 MAE 或 RMSE 上因加入新闻变量而改善。")
else:
    robust_lines.append("- 未得到有效稳健性模型结果，主要原因可能是辅助目标或特定 lag 特征缺失。")

robust_conclusion = "如果多个稳健性设定下方向一致，可以说结论具有一定稳健性；若不同设定差异较大，应归因于样本量有限、目标变量缺失、新闻噪声或科技裁员事件的突发性。"
robust_interp = "# 稳健性检验解释\n\n" + "\n".join(robust_lines) + "\n\n" + robust_conclusion
safe_write_text(TEXT_DIR / "robustness_interpretation.md", robust_interp)

display(robustness_results)
print(robust_interp)


                   robustness_check                               target   feature_set         model status          MAE         RMSE  MAPE_nonzero     sMAPE  Poisson_Deviance  Directional_Accuracy
0   alternative_target_known_layoff  target_next_week_known_layoff_count    Macro Only         Ridge     ok  2151.020578  5769.923365      0.912714  1.702441      23370.193258              0.615385
1   alternative_target_known_layoff  target_next_week_known_layoff_count    Macro Only  RandomForest     ok  2093.361434  5716.354636      0.860769  1.495003      14454.687899              0.615385
2   alternative_target_known_layoff  target_next_week_known_layoff_count  Macro + News         Ridge     ok  2241.188707  5868.388832      1.322627  1.609240      26215.283064              0.538462
3   alternative_target_known_layoff  target_next_week_known_layoff_count  Macro + News  RandomForest     ok  2075.955791  5696.688495      0.774292  1.353537      12103.387346              0.584615
4         

## 第 10 部分：RQ3 AI vs 非 AI 异质性分析

本部分在存在 AI / 非 AI 每周裁员事件数时运行。包括趋势图、AI 与非 AI 各自的 TLCC，以及简单分组回归。解释时只讨论差异化滞后关联，不写因果结论。


In [12]:
# ==============================
# 第 10 部分：RQ3 AI vs 非 AI 异质性分析
# ==============================

hetero_rows = []
hetero_tlcc_rows = []

ai_col = "weekly_ai_layoff_event_count"
nonai_col = "weekly_non_ai_layoff_event_count"
hetero_available = ai_col in df.columns and nonai_col in df.columns

if hetero_available:
    # 趋势图。
    plt.figure(figsize=(11, 5))
    plt.plot(df[time_col], pd.to_numeric(df[ai_col], errors="coerce"), label="AI layoffs", marker="o", markersize=3)
    plt.plot(df[time_col], pd.to_numeric(df[nonai_col], errors="coerce"), label="Non-AI layoffs", marker="o", markersize=3)
    plt.title("AI vs Non-AI weekly layoff event counts")
    plt.xlabel("Time")
    plt.ylabel("Event count")
    plt.legend()
    plot_save_close(FIG_DIR / "fig5_ai_vs_nonai_trend.png")

    # 创建下一周 AI / 非 AI 目标。
    ai_target = "target_next_week_ai_layoff_event_count"
    nonai_target = "target_next_week_non_ai_layoff_event_count"
    ai_log_target = "target_next_week_log_ai_layoff_event_count"
    nonai_log_target = "target_next_week_log_non_ai_layoff_event_count"
    if ai_target not in df.columns:
        df[ai_target] = pd.to_numeric(df[ai_col], errors="coerce").shift(-1)
    if nonai_target not in df.columns:
        df[nonai_target] = pd.to_numeric(df[nonai_col], errors="coerce").shift(-1)
    df[ai_log_target] = np.log1p(pd.to_numeric(df[ai_target], errors="coerce"))
    df[nonai_log_target] = np.log1p(pd.to_numeric(df[nonai_target], errors="coerce"))

    # AI / 非 AI TLCC：新闻 lag1/lag2 对两个目标分别计算。
    for outcome_name, outcome_col in [("AI", ai_col), ("Non-AI", nonai_col)]:
        y0 = pd.to_numeric(df[outcome_col], errors="coerce")
        for var in tlcc_vars:
            x0 = pd.to_numeric(df[var], errors="coerce")
            for lag in [1, 2]:
                pair = pd.DataFrame({"x": x0.shift(lag), "y": y0}).dropna()
                if len(pair) >= 3 and pair["x"].std() > 0 and pair["y"].std() > 0:
                    corr, pval = stats.pearsonr(pair["x"], pair["y"])
                else:
                    corr, pval = np.nan, np.nan
                hetero_tlcc_rows.append({"group": outcome_name, "variable": var, "lag": lag, "correlation": corr, "p_value": pval, "n_obs": len(pair), "significance": sig_star(pval)})

    hetero_tlcc = pd.DataFrame(hetero_tlcc_rows)
    hetero_tlcc.to_csv(RESULT_DIR / "heterogeneity_tlcc_ai_nonai.csv", index=False, encoding="utf-8-sig")

    # 分组回归：下一周 AI / 非 AI log 事件数 ~ 新闻 lag1/lag2 + AR 控制。
    hetero_reg_features = numeric_existing_features(df, core_news_lag1 + core_news_lag2 + ar_controls)
    for group_name, y_col in [("AI", ai_log_target), ("Non-AI", nonai_log_target)]:
        reg_df = df[[y_col] + hetero_reg_features].copy()
        for c in [y_col] + hetero_reg_features:
            reg_df[c] = pd.to_numeric(reg_df[c], errors="coerce")
        reg_df = reg_df.dropna()
        if len(reg_df) < max(10, len(hetero_reg_features) + 5) or not hetero_reg_features:
            hetero_rows.append({"group": group_name, "model": "heterogeneity_OLS_HAC", "status": f"skipped_small_sample_or_no_features_n={len(reg_df)}"})
            continue
        try:
            Xh = sm.add_constant(reg_df[hetero_reg_features], has_constant="add")
            yh = reg_df[y_col]
            fit = sm.OLS(yh, Xh).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
            for term in fit.params.index:
                hetero_rows.append({
                    "group": group_name,
                    "model": "heterogeneity_OLS_HAC",
                    "status": "ok",
                    "term": term,
                    "coef": fit.params.get(term, np.nan),
                    "std_err": fit.bse.get(term, np.nan),
                    "stat": fit.tvalues.get(term, np.nan),
                    "p_value": fit.pvalues.get(term, np.nan),
                    "significance": sig_star(fit.pvalues.get(term, np.nan)),
                    "n_obs": int(fit.nobs),
                    "r_squared": fit.rsquared,
                })
        except Exception as e:
            hetero_rows.append({"group": group_name, "model": "heterogeneity_OLS_HAC", "status": "failed", "error": str(e)})
else:
    hetero_rows.append({"group": "AI_vs_NonAI", "status": "skipped_missing_weekly_ai_or_nonai_layoff_event_count"})
    hetero_tlcc = pd.DataFrame(columns=["group", "variable", "lag", "correlation", "p_value", "n_obs", "significance"])
    hetero_tlcc.to_csv(RESULT_DIR / "heterogeneity_tlcc_ai_nonai.csv", index=False, encoding="utf-8-sig")

heterogeneity_results = pd.DataFrame(hetero_rows)
heterogeneity_results.to_csv(RESULT_DIR / "heterogeneity_results.csv", index=False, encoding="utf-8-sig")

# 自动解释。
if hetero_available and not hetero_tlcc.empty:
    ai_best = hetero_tlcc[hetero_tlcc["group"] == "AI"].dropna(subset=["correlation"]).copy()
    nonai_best = hetero_tlcc[hetero_tlcc["group"] == "Non-AI"].dropna(subset=["correlation"]).copy()
    def best_line(sub, label):
        if sub.empty:
            return f"{label} 组没有足够 TLCC 结果。"
        sub = sub.assign(abs_corr=sub["correlation"].abs()).sort_values("abs_corr", ascending=False)
        r = sub.iloc[0]
        return f"{label} 组最高相关为 `{r['variable']}` lag{int(r['lag'])}，corr={r['correlation']:.3f}, p={r['p_value']:.3f}{r['significance']}。"
    hetero_text = best_line(ai_best, "AI") + "\n" + best_line(nonai_best, "非 AI")
    hetero_conclusion = "若非 AI 结果更稳定显著，可以解释为非 AI 裁员事件与媒体裁员新闻强度之间的滞后关系更稳定；AI 裁员可能更突发，或受公司战略调整、融资周期和技术路线变化影响更大。"
else:
    hetero_text = "数据中缺少 AI / 非 AI 每周裁员事件数，或有效观测不足，因此跳过异质性分析。"
    hetero_conclusion = "这不代表异质性不存在，只说明当前建模矩阵无法支持该检验。"

hetero_interp = f"""# AI vs 非 AI 异质性解释

{hetero_text}

{hetero_conclusion}

本部分为描述性和分组回归分析，不用于证明因果关系。
"""
safe_write_text(TEXT_DIR / "heterogeneity_interpretation.md", hetero_interp)

display(heterogeneity_results)
print(hetero_interp)


     group                  model status                            term      coef   std_err      stat       p_value significance  n_obs  r_squared
0       AI  heterogeneity_OLS_HAC     ok                           const  0.698023  0.355963  1.960939  4.988612e-02           **    216   0.061687
1       AI  heterogeneity_OLS_HAC     ok   weekly_layoff_news_count_lag1  0.003404  0.001566  2.173424  2.974843e-02           **    216   0.061687
2       AI  heterogeneity_OLS_HAC     ok   weekly_layoff_news_share_lag1  0.201053  1.177625  0.170727  8.644383e-01                 216   0.061687
3       AI  heterogeneity_OLS_HAC     ok      weekly_negative_share_lag1 -0.549772  0.944610 -0.582009  5.605606e-01                 216   0.061687
4       AI  heterogeneity_OLS_HAC     ok            sentiment_shock_lag1 -0.104980  0.961181 -0.109220  9.130283e-01                 216   0.061687
5       AI  heterogeneity_OLS_HAC     ok   weekly_layoff_news_count_lag2  0.000646  0.001471  0.439119  6.605753

## 第 11 部分：LSTM 扩展模型，可选

LSTM 不是主结论。只有在 TensorFlow/Keras 已安装、样本量足够、特征缺失处理完成且不会明显拖慢 Notebook 时才运行；否则自动生成“不采用 LSTM 的理由”。


In [13]:
# ==============================
# 第 11 部分：LSTM 扩展模型（可选）
# ==============================

lstm_ran = False
lstm_note_path = TEXT_DIR / "lstm_optional_note.md"

# 周度数据通常样本较少。这里要求至少 120 个训练样本，避免 LSTM 明显过拟合。
can_try_lstm = TENSORFLOW_AVAILABLE and len(train_df) >= 120 and len(test_df) >= 12 and len(macro_news_features) > 0

if can_try_lstm:
    try:
        T = 4
        X_all = df[macro_news_features].copy()
        for c in X_all.columns:
            X_all[c] = pd.to_numeric(X_all[c], errors="coerce")
        y_all = np.log1p(np.clip(pd.to_numeric(df[main_target], errors="coerce"), 0, None))

        # 使用训练期中位数填补，并进行标准化。注意只用训练期拟合 imputer/scaler，避免信息泄漏。
        train_indices = train_df.index
        test_indices = test_df.index
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        X_train_base = X_all.loc[train_indices]
        X_test_base = X_all.loc[test_indices]
        imputer.fit(X_train_base)
        scaler.fit(imputer.transform(X_train_base))
        X_scaled = pd.DataFrame(scaler.transform(imputer.transform(X_all)), index=df.index, columns=macro_news_features)

        # 创建滑动窗口：用过去 T 周特征预测当前行的下一周目标。
        X_seq, y_seq, idx_seq = [], [], []
        for i in range(T - 1, len(df)):
            if pd.isna(y_all.iloc[i]):
                continue
            X_seq.append(X_scaled.iloc[i-T+1:i+1].values)
            y_seq.append(y_all.iloc[i])
            idx_seq.append(df.index[i])
        X_seq = np.asarray(X_seq)
        y_seq = np.asarray(y_seq)
        idx_seq = np.asarray(idx_seq)

        train_seq_mask = np.isin(idx_seq, train_indices)
        test_seq_mask = np.isin(idx_seq, test_indices)
        X_seq_train, y_seq_train = X_seq[train_seq_mask], y_seq[train_seq_mask]
        X_seq_test, y_seq_test_log = X_seq[test_seq_mask], y_seq[test_seq_mask]
        y_seq_test_raw = np.expm1(y_seq_test_log)

        if len(X_seq_train) >= 80 and len(X_seq_test) >= 8:
            tf.random.set_seed(RANDOM_STATE)
            model = Sequential([
                LSTM(16, input_shape=(T, X_seq_train.shape[2])),
                Dropout(0.10),
                Dense(1),
            ])
            model.compile(optimizer="adam", loss="mse")
            es = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
            model.fit(X_seq_train, y_seq_train, epochs=80, batch_size=16, validation_split=0.2, callbacks=[es], verbose=0)
            pred_log = model.predict(X_seq_test, verbose=0).ravel()
            pred = np.expm1(pred_log)
            metrics = evaluate_predictions(y_seq_test_raw, pred)
            lstm_result = pd.DataFrame([{ "model": "LSTM_T4", "status": "ok", **metrics }])
            lstm_result.to_csv(RESULT_DIR / "lstm_optional_results.csv", index=False, encoding="utf-8-sig")
            lstm_ran = True
            safe_write_text(lstm_note_path, "# LSTM 扩展模型说明\n\n本次已运行 T=4 的简单 LSTM 扩展模型。由于周度样本有限，LSTM 仅作为补充比较，不作为主结论。")
            display(lstm_result)
        else:
            raise ValueError(f"滑动窗口后样本不足：train={len(X_seq_train)}, test={len(X_seq_test)}")
    except Exception as e:
        reason = f"LSTM 未运行或运行失败：{e}"
        safe_write_text(lstm_note_path, f"# 不采用 LSTM 的理由\n\n{reason}\n\n由于周度样本量有限，LSTM 容易过拟合，因此只作为扩展方向，不作为主结论。")
        print(reason)
else:
    reasons = []
    if not TENSORFLOW_AVAILABLE:
        reasons.append("TensorFlow/Keras 未安装")
    if len(train_df) < 120:
        reasons.append(f"训练样本不足 120（当前 {len(train_df)}）")
    if len(test_df) < 12:
        reasons.append(f"测试样本不足 12（当前 {len(test_df)}）")
    if len(macro_news_features) == 0:
        reasons.append("Macro + News 特征为空")
    reason = "；".join(reasons) if reasons else "运行条件不足"
    safe_write_text(lstm_note_path, f"# 不采用 LSTM 的理由\n\n{reason}。\n\n由于周度样本量有限，LSTM 容易过拟合，因此只作为扩展方向，不作为主结论。")
    print("LSTM 未运行：", reason)


LSTM 未运行： TensorFlow/Keras 未安装


## 第 12 部分：自动生成报告可用文字

本部分汇总前面所有结果，生成中文 Markdown 文件，方便直接复制到课程报告中。文字会保持谨慎表述，不使用“证明因果关系”等过度结论。


In [14]:
# ==============================
# 第 12 部分：自动生成报告可用文字
# ==============================

def read_text_if_exists(path, fallback=""):
    path = Path(path)
    if path.exists():
        return path.read_text(encoding="utf-8")
    return fallback

# 从已生成的解释文件中抽取内容，形成最终汇总。
tlcc_text = read_text_if_exists(TEXT_DIR / "tlcc_interpretation.md", "TLCC 结果未生成。")
reg_text = read_text_if_exists(TEXT_DIR / "regression_interpretation.md", "回归结果未生成。")
model_text = read_text_if_exists(TEXT_DIR / "model_comparison_interpretation.md", "模型比较结果未生成。")
ablation_text = read_text_if_exists(TEXT_DIR / "ablation_interpretation.md", "消融实验结果未生成。")
robust_text = read_text_if_exists(TEXT_DIR / "robustness_interpretation.md", "稳健性结果未生成。")
hetero_text = read_text_if_exists(TEXT_DIR / "heterogeneity_interpretation.md", "异质性结果未生成。")
lstm_text = read_text_if_exists(TEXT_DIR / "lstm_optional_note.md", "LSTM 说明未生成。")

# 额外提炼模型比较中的最佳结果。
if 'model_comparison' in globals() and not model_comparison.empty:
    best = model_comparison.iloc[0]
    best_summary = f"样本外预测中，最低 MAE 的组合为 {best['feature_set']} / {best['model']}，MAE={best['MAE']:.3f}，RMSE={best['RMSE']:.3f}。"
else:
    best_summary = "由于可用特征或样本限制，未得到完整的样本外模型比较结果。"

# Macro + News 是否优于 Macro Only 的整体判断。
if 'ablation_table' in globals() and not ablation_table.empty:
    mae_improved = int((ablation_table["improvement_mae_pct"] > 0).sum())
    rmse_improved = int((ablation_table["improvement_rmse_pct"] > 0).sum())
    total_cmp = len(ablation_table)
    if mae_improved == total_cmp and rmse_improved == total_cmp:
        macro_news_sentence = "Macro + News 在全部可比较模型中均降低了 MAE 和 RMSE，说明新闻变量具有较稳定的样本外预测增量。"
    elif mae_improved > 0 or rmse_improved > 0:
        macro_news_sentence = f"Macro + News 在部分模型中改善预测误差（MAE 改善 {mae_improved}/{total_cmp}，RMSE 改善 {rmse_improved}/{total_cmp}），说明新闻变量可能提供一定预测增量，但效果并非在所有模型中稳定。"
    else:
        macro_news_sentence = "Macro + News 未能稳定优于 Macro Only，新闻变量更可能体现机制解释价值，而非稳定预测优势。"
else:
    macro_news_sentence = "由于缺少成对消融结果，暂不能判断 Macro + News 是否稳定优于 Macro Only。"

summary = f"""# 成员B结果摘要：模型、机制检验与显著性分析

## 1. 成员B分析目标

成员B围绕 proposal 中的 RQ1、RQ2 与 RQ3 展开分析。核心任务不是简单堆叠模型，而是检验新闻情绪和裁员相关新闻强度是否具有前瞻信息，并比较在控制宏观劳动力指标后，新闻变量是否能提升下一周科技裁员事件数的样本外预测表现。同时，本部分也对 AI 与非 AI 公司裁员事件进行描述性异质性分析。

## 2. RQ1：TLCC 时滞交叉相关结果

{tlcc_text}

## 3. RQ1：HAC / Poisson / Negative Binomial 回归结果

{reg_text}

## 4. RQ2：Macro Only、News Only、Macro + News 模型对比

{best_summary}

{model_text}

## 5. 消融实验结论

{macro_news_sentence}

{ablation_text}

## 6. 稳健性检验结论

{robust_text}

## 7. AI vs 非 AI 异质性分析结论

{hetero_text}

## 8. LSTM 扩展模型说明

{lstm_text}

## 9. 局限性

第一，周度样本量有限，尤其在加入多个滞后变量和宏观控制后，自由度会下降，显著性结果可能不稳定。第二，新闻变量可能同时反映真实裁员事件、市场预期和媒体报道偏好，因此即使出现滞后关联，也不能解释为因果关系。第三，科技裁员事件具有突发性和公司层面战略因素，部分事件可能难以由宏观指标或新闻情绪稳定预测。第四，若成员A建模矩阵中某些 lag、rolling 或 AI/非 AI 分组变量缺失，相关检验会自动跳过，结论范围也应相应收窄。

## 10. 可放入报告的简短总结段落

本研究的成员B部分通过 TLCC、HAC 稳健回归、计数模型和样本外预测模型，检验新闻情绪与科技裁员事件之间的滞后关联。结果应被解释为“新闻变量可能具有一定前瞻信息”或“存在滞后关联”，而不是因果证明。在预测任务中，Macro + News 是否优于 Macro Only 取决于具体模型和指标；若提升不稳定，则说明新闻变量更可能体现机制解释价值，而非稳定预测优势。AI 与非 AI 异质性分析进一步提供了描述性证据：不同类型公司裁员对新闻强度和情绪的响应可能存在差异，但该部分不做因果推断。
"""

safe_write_text(TEXT_DIR / "memberB_results_summary_for_report.md", summary)
print("已生成报告摘要：", TEXT_DIR / "memberB_results_summary_for_report.md")
print(summary[:2000])


已生成报告摘要： /Users/peiyifan/Desktop/机器学习/05_report_text/memberB_results_summary_for_report.md
# 成员B结果摘要：模型、机制检验与显著性分析

## 1. 成员B分析目标

成员B围绕 proposal 中的 RQ1、RQ2 与 RQ3 展开分析。核心任务不是简单堆叠模型，而是检验新闻情绪和裁员相关新闻强度是否具有前瞻信息，并比较在控制宏观劳动力指标后，新闻变量是否能提升下一周科技裁员事件数的样本外预测表现。同时，本部分也对 AI 与非 AI 公司裁员事件进行描述性异质性分析。

## 2. RQ1：TLCC 时滞交叉相关结果

# TLCC 时滞交叉相关解释

使用 weekly_layoff_event_count 作为 TLCC 目标变量。

本部分计算新闻变量在 `t-k` 周与裁员事件数在 `t` 周之间的相关系数。`lag=1` 表示新闻变量领先裁员事件 1 周，`lag=2` 表示领先 2 周，`lag<0` 表示新闻变量滞后于裁员事件。

- 使用的新闻变量：weekly_layoff_news_count、weekly_layoff_news_share、weekly_negative_share、sentiment_shock、weekly_avg_sentiment
- lag1 上相关最高的变量是 `weekly_layoff_news_count`，相关系数为 0.243，p 值为 0.000***。
- 这表明新闻变量与之后裁员事件存在一定滞后关联，具有前瞻信息迹象。

需要注意，TLCC 是双变量相关分析，未控制宏观变量和历史裁员状态，因此只能说明滞后关联方向，不能证明因果关系。


## 3. RQ1：HAC / Poisson / Negative Binomial 回归结果

# 滞后回归解释

本部分以 `target_next_week_log_layoff_event_count` 作为 OLS 因变量，并使用 Newey-West / HAC 稳健标准误（maxlags=4）。Poisson GLM 使用 `target_next_week_layoff_event_count` 作为计数因变量；Negative Binomial 在可


## 修复后完整重跑：变量筛选、RQ1/RQ2/RQ3 与质量检查

本章节修复旧版 `is_forbidden_feature` 中误用 `"y_" in col` 的问题，并重新生成所有带 `_fixed` 后缀的表格、图表和文字解释。重点保证 `weekly_` 开头的新闻变量、历史滞后变量和 AI/非 AI 变量不会被误删，同时严格禁止 `target_next_week_*` 等未来标签进入模型特征。


In [15]:

# ==============================
# FIXED PIPELINE AFTER FORBIDDEN FEATURE BUG
# 修复后完整重跑：变量分组、TLCC、回归、预测、消融、特征重要性、稳健性、异质性、质量检查
# ==============================

print("开始执行修复后完整重跑流程...")

# ---------- 1. 安全特征禁用逻辑 ----------
def is_forbidden_feature_fixed(col):
    """只排除真正的标签列和未来信息列，不误伤 weekly_ 变量。"""
    low = str(col).lower()
    forbidden_keywords = ["target_next_week", "next_week", "future", "lead", "t_plus", "label"]
    if low == "y" or low.startswith("y_"):
        return True
    return any(k in low for k in forbidden_keywords)

# 覆盖全局函数，保证后续 Notebook 单元也使用修复版。
is_forbidden_feature = is_forbidden_feature_fixed

# 若根目录没有数据，但成员A输出目录有数据，则复制一份到根目录，保证读取逻辑一致。
root_matrix = BASE_DIR / "final_model_matrix_v1.csv"
a_output_matrix = BASE_DIR / "data final" / "A_outputs" / "final_model_matrix_v1.csv"
if not root_matrix.exists() and a_output_matrix.exists():
    import shutil
    shutil.copy2(a_output_matrix, root_matrix)
    print(f"已从成员A输出目录复制建模矩阵到：{root_matrix}")

# 重新读取数据，避免受前面旧结果状态影响。
candidate_files_fixed = [
    BASE_DIR / "final_model_matrix_v1.csv",
    BASE_DIR / "final_model_matrix.csv",
    BASE_DIR / "final_weekly_panel.csv",
    BASE_DIR / "final_weekly_panel_v0.csv",
    BASE_DIR / "data final" / "A_outputs" / "final_model_matrix_v1.csv",
]
selected_file_fixed = next((p for p in candidate_files_fixed if p.exists()), None)
if selected_file_fixed is None:
    raise FileNotFoundError("未找到建模数据文件，请先运行成员A数据 pipeline。")

df_fixed = pd.read_csv(selected_file_fixed, low_memory=False)
time_candidates_fixed = ["week", "date", "Week", "week_start", "start_week"]
time_col_fixed = next((c for c in time_candidates_fixed if c in df_fixed.columns), None)
if time_col_fixed is None:
    raise ValueError("未找到时间列。")
df_fixed[time_col_fixed] = pd.to_datetime(df_fixed[time_col_fixed], errors="coerce")
df_fixed = df_fixed.dropna(subset=[time_col_fixed]).sort_values(time_col_fixed).reset_index(drop=True)

# 尽量转换数值列。
for c in df_fixed.columns:
    if c == time_col_fixed:
        continue
    if df_fixed[c].dtype == "object":
        tmp = pd.to_numeric(df_fixed[c], errors="coerce")
        if tmp.notna().sum() >= max(1, int(0.5 * df_fixed[c].notna().sum())):
            df_fixed[c] = tmp

main_target_fixed = "target_next_week_layoff_event_count"
main_log_target_fixed = "target_next_week_log_layoff_event_count"
if main_target_fixed not in df_fixed.columns:
    if "weekly_layoff_event_count" in df_fixed.columns:
        df_fixed[main_target_fixed] = pd.to_numeric(df_fixed["weekly_layoff_event_count"], errors="coerce").shift(-1)
    else:
        raise ValueError("缺少主目标，且无法由 weekly_layoff_event_count 构造。")
if main_log_target_fixed not in df_fixed.columns:
    df_fixed[main_log_target_fixed] = np.log1p(pd.to_numeric(df_fixed[main_target_fixed], errors="coerce"))
df_fixed[main_target_fixed] = pd.to_numeric(df_fixed[main_target_fixed], errors="coerce")
df_fixed[main_log_target_fixed] = pd.to_numeric(df_fixed[main_log_target_fixed], errors="coerce")
df_fixed = df_fixed.dropna(subset=[main_target_fixed]).reset_index(drop=True)

# ---------- 2. 修复后变量分组 ----------
def clean_feature_list_fixed(cols, df_columns=None):
    seen, out = set(), []
    for c in cols:
        if c is None or c in seen:
            continue
        if df_columns is not None and c not in df_columns:
            continue
        seen.add(c)
        out.append(c)
    return out

def numeric_existing_features_fixed(data, features):
    valid = []
    for c in clean_feature_list_fixed(features, data.columns):
        if is_forbidden_feature_fixed(c):
            continue
        s = data[c]
        if pd.api.types.is_numeric_dtype(s) or pd.to_numeric(s, errors="coerce").notna().sum() > 0:
            valid.append(c)
    return valid

all_cols_fixed = list(df_fixed.columns)
all_lower_fixed = {c: c.lower() for c in all_cols_fixed}

base_news_fixed = [
    "weekly_news_count",
    "weekly_layoff_news_count",
    "weekly_layoff_news_share",
    "weekly_negative_share",
    "weekly_avg_sentiment",
    "weekly_layoff_avg_sentiment",
    "sentiment_shock",
    "has_news_week",
]
news_priority_fixed = []
for b in base_news_fixed:
    news_priority_fixed.append(b)
    for k in range(1, 5):
        news_priority_fixed.append(f"{b}_lag{k}")
    news_priority_fixed.append(f"{b}_roll4_mean")
    news_priority_fixed.append(f"{b}_roll4_sum")
news_keywords_fixed = ["news", "sentiment", "negative", "layoff_news", "hiring_news"]
news_auto_fixed = [c for c in all_cols_fixed if any(k in all_lower_fixed[c] for k in news_keywords_fixed)]
news_features_fixed = numeric_existing_features_fixed(df_fixed, news_priority_fixed + news_auto_fixed)

macro_priority_fixed = [
    "initial_jobless_claims_k",
    "claims_4w_avg",
    "unemployment_rate",
    "jolts_job_openings_k",
    "information_sector_emp_k",
    "computer_math_emp_k",
]
macro_keywords_fixed = ["claims", "unemployment", "jolts", "openings", "employment", "emp", "labor", "gdp", "inflation"]
macro_auto_fixed = []
for c in all_cols_fixed:
    low = all_lower_fixed[c]
    if any(k in low for k in macro_keywords_fixed) and not any(k in low for k in news_keywords_fixed):
        macro_auto_fixed.append(c)
macro_features_fixed = numeric_existing_features_fixed(df_fixed, macro_priority_fixed + macro_auto_fixed)

ar_priority_fixed = [
    "weekly_layoff_event_count_lag1",
    "weekly_layoff_event_count_lag2",
    "weekly_layoff_event_count_lag3",
    "weekly_layoff_event_count_lag4",
    "weekly_layoff_event_count_roll4_mean",
    "weekly_layoff_event_count_roll4_sum",
]
ar_auto_fixed = [c for c in all_cols_fixed if ("layoff_event_count_lag" in all_lower_fixed[c] or "layoff_event_count_roll" in all_lower_fixed[c])]
ar_features_fixed = numeric_existing_features_fixed(df_fixed, ar_priority_fixed + ar_auto_fixed)
if not ar_features_fixed and "weekly_layoff_event_count" in df_fixed.columns:
    ar_features_fixed = numeric_existing_features_fixed(df_fixed, ["weekly_layoff_event_count"])

heterogeneity_priority_fixed = [
    "weekly_ai_layoff_event_count",
    "weekly_non_ai_layoff_event_count",
    "weekly_ai_layoff_event_count_lag1",
    "weekly_non_ai_layoff_event_count_lag1",
]
heterogeneity_auto_fixed = [c for c in all_cols_fixed if ("ai_layoff" in all_lower_fixed[c] or "non_ai_layoff" in all_lower_fixed[c])]
heterogeneity_features_fixed = numeric_existing_features_fixed(df_fixed, heterogeneity_priority_fixed + heterogeneity_auto_fixed)

forbidden_features_fixed = [c for c in all_cols_fixed if is_forbidden_feature_fixed(c)]

# 核心变量诊断：如果存在但没进组，记录警告，并尽量修复分组。
core_should_include = [
    "weekly_layoff_news_count",
    "weekly_layoff_news_count_lag1",
    "weekly_layoff_news_count_lag2",
    "weekly_layoff_news_share",
    "weekly_layoff_news_share_lag1",
    "weekly_layoff_news_share_lag2",
    "weekly_negative_share",
    "weekly_negative_share_lag1",
    "weekly_negative_share_lag2",
    "sentiment_shock_lag1",
    "sentiment_shock_lag2",
    "weekly_layoff_event_count_lag1",
    "weekly_layoff_event_count_lag2",
    "weekly_layoff_event_count_roll4_mean",
]

# 强制补回应在新闻组/AR组的核心变量。
for c in core_should_include:
    if c in df_fixed.columns and not is_forbidden_feature_fixed(c):
        if any(key in c for key in ["news", "negative", "sentiment"]):
            if c not in news_features_fixed and pd.to_numeric(df_fixed[c], errors="coerce").notna().sum() > 0:
                news_features_fixed.append(c)
        if "layoff_event_count" in c:
            if c not in ar_features_fixed and pd.to_numeric(df_fixed[c], errors="coerce").notna().sum() > 0:
                ar_features_fixed.append(c)

news_features_fixed = clean_feature_list_fixed(news_features_fixed, df_fixed.columns)
macro_features_fixed = clean_feature_list_fixed(macro_features_fixed, df_fixed.columns)
ar_features_fixed = clean_feature_list_fixed(ar_features_fixed, df_fixed.columns)
heterogeneity_features_fixed = clean_feature_list_fixed(heterogeneity_features_fixed, df_fixed.columns)

# 保存变量组。
feature_rows_fixed = []
for group, feats in [
    ("news_features", news_features_fixed),
    ("macro_features", macro_features_fixed),
    ("ar_features", ar_features_fixed),
    ("heterogeneity_features", heterogeneity_features_fixed),
    ("forbidden_features", forbidden_features_fixed),
]:
    for f in feats:
        feature_rows_fixed.append({"feature_group": group, "feature": f})
feature_groups_fixed_df = pd.DataFrame(feature_rows_fixed)
feature_groups_fixed_df.to_csv(RESULT_DIR / "memberB_feature_groups_fixed.csv", index=False, encoding="utf-8-sig")

# 变量诊断表。
def assigned_group_fixed(col):
    groups = []
    if col in news_features_fixed: groups.append("news_features")
    if col in macro_features_fixed: groups.append("macro_features")
    if col in ar_features_fixed: groups.append("ar_features")
    if col in heterogeneity_features_fixed: groups.append("heterogeneity_features")
    if col in forbidden_features_fixed: groups.append("forbidden_features")
    return ";".join(groups) if groups else "unassigned"

diag_cols = clean_feature_list_fixed(core_should_include + forbidden_features_fixed + news_features_fixed[:10] + ar_features_fixed[:10], all_cols_fixed)
diag_rows = []
for col in diag_cols:
    exists = col in df_fixed.columns
    should = col in core_should_include
    group = assigned_group_fixed(col)
    is_forb = is_forbidden_feature_fixed(col)
    note = ""
    if should and exists and group == "unassigned":
        note = "WARNING: 核心变量存在但未进入变量组"
    elif should and exists and is_forb:
        note = "ERROR: 核心变量被误判为 forbidden"
    elif should and exists:
        note = "OK: 核心变量已纳入对应变量组"
    elif is_forb:
        note = "OK: 标签或未来信息列被排除"
    diag_rows.append({
        "column": col,
        "exists_in_data": exists,
        "assigned_group": group,
        "is_forbidden": is_forb,
        "should_be_included": should,
        "note": note,
    })
feature_diag_fixed = pd.DataFrame(diag_rows)
feature_diag_fixed.to_csv(RESULT_DIR / "memberB_feature_diagnostic_after_fix.csv", index=False, encoding="utf-8-sig")

print("修复后变量组数量：")
print("news_features_fixed =", len(news_features_fixed))
print("macro_features_fixed =", len(macro_features_fixed))
print("ar_features_fixed =", len(ar_features_fixed))
print("heterogeneity_features_fixed =", len(heterogeneity_features_fixed))
print("forbidden_features_fixed =", len(forbidden_features_fixed))
print("核心变量诊断：")
display(feature_diag_fixed[feature_diag_fixed["should_be_included"]])

# ---------- 3. 时间切分 ----------
train_mask_default_fixed = (df_fixed[time_col_fixed] >= pd.Timestamp("2022-01-03")) & (df_fixed[time_col_fixed] <= pd.Timestamp("2024-12-31"))
test_mask_default_fixed = (df_fixed[time_col_fixed] >= pd.Timestamp("2025-01-01")) & (df_fixed[time_col_fixed] <= pd.Timestamp("2026-04-13"))
if train_mask_default_fixed.sum() >= 20 and test_mask_default_fixed.sum() >= 5:
    train_df_fixed = df_fixed.loc[train_mask_default_fixed].copy()
    test_df_fixed = df_fixed.loc[test_mask_default_fixed].copy()
    split_method_fixed = "calendar_default"
else:
    split_idx_fixed = int(len(df_fixed) * 0.75)
    split_idx_fixed = max(1, min(split_idx_fixed, len(df_fixed) - 1))
    train_df_fixed = df_fixed.iloc[:split_idx_fixed].copy()
    test_df_fixed = df_fixed.iloc[split_idx_fixed:].copy()
    split_method_fixed = "fallback_75_25"
print("修复后时间切分：", split_method_fixed, len(train_df_fixed), len(test_df_fixed))

# ---------- 4. 通用建模函数 ----------
def evaluate_predictions_fixed(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    nz = y_true > 0
    mape_nonzero = mean_absolute_percentage_error(y_true[nz], y_pred[nz]) if nz.sum() else np.nan
    denom = np.abs(y_true) + np.abs(y_pred)
    smape_val = np.mean(2 * np.abs(y_pred[denom > 0] - y_true[denom > 0]) / denom[denom > 0]) if (denom > 0).sum() else np.nan
    if len(y_true) >= 2:
        da = np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred)))
    else:
        da = np.nan
    try:
        pdev = mean_poisson_deviance(np.clip(y_true, 0, None), np.clip(y_pred, 1e-9, None))
    except Exception:
        pdev = np.nan
    return {"MAE": mae, "RMSE": rmse, "MAPE_nonzero": mape_nonzero, "sMAPE": smape_val, "Poisson_Deviance": pdev, "Directional_Accuracy": da}

def prepare_xy_fixed(data, features, target):
    feats = numeric_existing_features_fixed(data, features)
    X = data[feats].copy()
    for c in feats:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    y = pd.to_numeric(data[target], errors="coerce")
    mask = y.notna()
    return X.loc[mask], y.loc[mask], feats, mask

def sig_star_fixed(p):
    if pd.isna(p): return ""
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return ""

# ---------- 5. RQ1 TLCC fixed ----------
tlcc_target_fixed = "weekly_layoff_event_count" if "weekly_layoff_event_count" in df_fixed.columns else main_target_fixed
tlcc_vars_fixed = [v for v in [
    "weekly_layoff_news_count",
    "weekly_layoff_news_share",
    "weekly_negative_share",
    "weekly_avg_sentiment",
    "weekly_layoff_avg_sentiment",
    "sentiment_shock",
] if v in df_fixed.columns and not is_forbidden_feature_fixed(v)]
if not any(v in tlcc_vars_fixed for v in ["weekly_layoff_news_count", "weekly_layoff_news_share"]):
    raise RuntimeError("TLCC 未包含 weekly_layoff_news_count 或 weekly_layoff_news_share，请检查变量分组。")

tlcc_rows_fixed = []
for var in tlcc_vars_fixed:
    x0 = pd.to_numeric(df_fixed[var], errors="coerce")
    y0 = pd.to_numeric(df_fixed[tlcc_target_fixed], errors="coerce")
    for lag in range(-4, 5):
        pair = pd.DataFrame({"x": x0.shift(lag), "y": y0}).dropna()
        if len(pair) >= 3 and pair["x"].std() > 0 and pair["y"].std() > 0:
            corr, pval = stats.pearsonr(pair["x"], pair["y"])
        else:
            corr, pval = np.nan, np.nan
        tlcc_rows_fixed.append({"variable": var, "lag": lag, "correlation": corr, "n_obs": len(pair), "p_value": pval, "significance": sig_star_fixed(pval)})
tlcc_fixed = pd.DataFrame(tlcc_rows_fixed)
tlcc_fixed.to_csv(RESULT_DIR / "tlcc_results_fixed.csv", index=False, encoding="utf-8-sig")

plt.figure(figsize=(10, 6))
for var in tlcc_vars_fixed:
    sub = tlcc_fixed[tlcc_fixed["variable"] == var]
    plt.plot(sub["lag"], sub["correlation"], marker="o", label=var)
plt.axhline(0, color="black", linewidth=0.8)
plt.axvline(0, color="gray", linestyle="--", linewidth=0.8)
plt.xlabel("Lag: positive means news leads layoffs")
plt.ylabel("Pearson correlation")
plt.title("Fixed TLCC between news variables and layoff events")
plt.legend(fontsize=8)
plot_save_close(FIG_DIR / "fig3_tlcc_lag_effect_fixed.png")

def tlcc_lag12_sentence(var):
    sub = tlcc_fixed[(tlcc_fixed["variable"] == var) & (tlcc_fixed["lag"].isin([1, 2]))].copy()
    if sub.empty or sub["correlation"].isna().all():
        return f"`{var}` 在 lag1/lag2 无足够观测。"
    sub["abs_corr"] = sub["correlation"].abs()
    r = sub.sort_values("abs_corr", ascending=False).iloc[0]
    sig = "达到 10% 显著性水平" if pd.notna(r["p_value"]) and r["p_value"] < 0.10 else "未达到 10% 显著性水平"
    return f"`{var}` 在 lag{int(r['lag'])} 的 lag1/lag2 中相关更强，corr={r['correlation']:.3f}, p={r['p_value']:.3f}{r['significance']}，{sig}。"

lead_sub_fixed = tlcc_fixed[tlcc_fixed["lag"].isin([1, 2])].dropna(subset=["correlation"]).copy()
if not lead_sub_fixed.empty:
    lead_sub_fixed["abs_corr"] = lead_sub_fixed["correlation"].abs()
    best_lead_fixed = lead_sub_fixed.sort_values("abs_corr", ascending=False).iloc[0]
    lead_support_fixed = "存在一定前瞻信息迹象" if pd.notna(best_lead_fixed["p_value"]) and best_lead_fixed["p_value"] < 0.10 else "前瞻关系的显著性证据不足"
else:
    best_lead_fixed = None
    lead_support_fixed = "有效 TLCC 结果不足"

tlcc_interp_fixed = f"""# TLCC 修复后解释

本次修复后，`weekly_` 开头的核心新闻变量不再被 `y_` 误删。TLCC 使用 `{tlcc_target_fixed}` 作为裁员事件目标。lag 的含义为：lag=1 表示 `news_{{t-1}}` 与 `layoff_t` 的相关，即新闻领先裁员 1 周；lag=2 表示新闻领先 2 周；lag=0 表示同周相关；lag<0 表示新闻滞后于裁员。

- 检验变量：{describe_available(tlcc_vars_fixed)}
- {tlcc_lag12_sentence('weekly_layoff_news_count')}
- {tlcc_lag12_sentence('weekly_layoff_news_share')}
- 综合 lag1/lag2 结果：{lead_support_fixed}。

如果 p 值没有达到 0.10 或 0.05，本报告只表述为探索性滞后关联，而不写成显著预测或因果关系。
"""
safe_write_text(TEXT_DIR / "tlcc_interpretation_fixed.md", tlcc_interp_fixed)
print("TLCC fixed variables:", tlcc_vars_fixed)
display(tlcc_fixed.head(30))

# ---------- 6. RQ1 回归 fixed ----------
core_news_lag1_fixed = numeric_existing_features_fixed(df_fixed, [
    "weekly_layoff_news_count_lag1",
    "weekly_layoff_news_share_lag1",
    "weekly_negative_share_lag1",
    "sentiment_shock_lag1",
])
core_news_lag2_fixed = numeric_existing_features_fixed(df_fixed, [
    "weekly_layoff_news_count_lag2",
    "weekly_layoff_news_share_lag2",
    "weekly_negative_share_lag2",
    "sentiment_shock_lag2",
])
ar_controls_fixed = numeric_existing_features_fixed(df_fixed, ["weekly_layoff_event_count_lag1", "weekly_layoff_event_count_lag2"])
macro_controls_fixed = macro_features_fixed[:8]
extra_controls_fixed = numeric_existing_features_fixed(df_fixed, ["has_news_week"])

reg_specs_fixed = {
    "R1_AR_only": ar_controls_fixed,
    "R2_AR_news_lag1": ar_controls_fixed + core_news_lag1_fixed + extra_controls_fixed,
    "R3_AR_news_lag12_macro": ar_controls_fixed + core_news_lag1_fixed + core_news_lag2_fixed + macro_controls_fixed + extra_controls_fixed,
}
reg_rows_fixed, reg_fit_rows_fixed = [], []
for model_name, feats in reg_specs_fixed.items():
    feats = numeric_existing_features_fixed(df_fixed, feats)
    reg_df = df_fixed[[main_log_target_fixed] + feats].copy()
    for c in [main_log_target_fixed] + feats:
        reg_df[c] = pd.to_numeric(reg_df[c], errors="coerce")
    reg_df = reg_df.dropna()
    if not feats or len(reg_df) < max(10, len(feats) + 5):
        reg_fit_rows_fixed.append({"model": model_name, "status": f"skipped_small_sample_or_no_features_n={len(reg_df)}"})
        continue
    try:
        X = sm.add_constant(reg_df[feats], has_constant="add")
        y = reg_df[main_log_target_fixed]
        fit = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
        reg_fit_rows_fixed.append({"model": model_name, "status": "ok", "n_obs": int(fit.nobs), "r_squared": fit.rsquared, "adj_r_squared": fit.rsquared_adj, "aic": fit.aic, "bic": fit.bic})
        for term in fit.params.index:
            reg_rows_fixed.append({"model": model_name, "model_type": "OLS_HAC", "term": term, "coef": fit.params[term], "std_err": fit.bse[term], "stat": fit.tvalues[term], "p_value": fit.pvalues[term], "significance": sig_star_fixed(fit.pvalues[term]), "n_obs": int(fit.nobs), "r_squared": fit.rsquared, "adj_r_squared": fit.rsquared_adj})
    except Exception as e:
        reg_fit_rows_fixed.append({"model": model_name, "status": "failed", "error": str(e)})

r3_feats_fixed = numeric_existing_features_fixed(df_fixed, reg_specs_fixed["R3_AR_news_lag12_macro"])
count_df_fixed = df_fixed[[main_target_fixed] + r3_feats_fixed].copy()
for c in [main_target_fixed] + r3_feats_fixed:
    count_df_fixed[c] = pd.to_numeric(count_df_fixed[c], errors="coerce")
count_df_fixed = count_df_fixed.dropna()
if r3_feats_fixed and len(count_df_fixed) >= max(10, len(r3_feats_fixed) + 5):
    Xc = sm.add_constant(count_df_fixed[r3_feats_fixed], has_constant="add")
    yc = np.clip(count_df_fixed[main_target_fixed], 0, None)
    try:
        pf = sm.GLM(yc, Xc, family=sm.families.Poisson()).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
        reg_fit_rows_fixed.append({"model": "R4_Poisson_R3_features", "status": "ok", "n_obs": int(pf.nobs), "aic": pf.aic})
        for term in pf.params.index:
            reg_rows_fixed.append({"model": "R4_Poisson_R3_features", "model_type": "Poisson_GLM_HAC", "term": term, "coef": pf.params[term], "std_err": pf.bse[term], "stat": pf.tvalues[term], "p_value": pf.pvalues[term], "significance": sig_star_fixed(pf.pvalues[term]), "n_obs": int(pf.nobs), "r_squared": np.nan, "adj_r_squared": np.nan})
    except Exception as e:
        reg_fit_rows_fixed.append({"model": "R4_Poisson_R3_features", "status": "failed", "error": str(e)})
    try:
        nbf = sm.GLM(yc, Xc, family=sm.families.NegativeBinomial()).fit(maxiter=200)
        reg_fit_rows_fixed.append({"model": "R5_NegativeBinomial_R3_features", "status": "ok", "n_obs": int(nbf.nobs), "aic": nbf.aic})
        for term in nbf.params.index:
            reg_rows_fixed.append({"model": "R5_NegativeBinomial_R3_features", "model_type": "NegativeBinomial_GLM", "term": term, "coef": nbf.params[term], "std_err": nbf.bse[term], "stat": nbf.tvalues[term], "p_value": nbf.pvalues[term], "significance": sig_star_fixed(nbf.pvalues[term]), "n_obs": int(nbf.nobs), "r_squared": np.nan, "adj_r_squared": np.nan})
    except Exception as e:
        reg_fit_rows_fixed.append({"model": "R5_NegativeBinomial_R3_features", "status": "failed_or_not_converged", "error": str(e)})
else:
    reg_fit_rows_fixed.append({"model": "R4_R5_count_models", "status": "skipped_no_features_or_small_sample"})

regression_results_fixed = pd.DataFrame(reg_rows_fixed)
regression_fit_fixed = pd.DataFrame(reg_fit_rows_fixed)
regression_results_fixed.to_csv(RESULT_DIR / "regression_results_fixed.csv", index=False, encoding="utf-8-sig")
regression_fit_fixed.to_csv(RESULT_DIR / "regression_model_fit_summary_fixed.csv", index=False, encoding="utf-8-sig")

def reg_term_sentence(term):
    sub = regression_results_fixed[(regression_results_fixed["term"] == term) & (regression_results_fixed["model"].isin(["R2_AR_news_lag1", "R3_AR_news_lag12_macro"]))]
    if sub.empty:
        return f"`{term}` 未进入可估计模型。"
    r = sub.iloc[0]
    direction = "正" if r["coef"] > 0 else "负"
    sig = "显著" if pd.notna(r["p_value"]) and r["p_value"] < 0.10 else "不显著"
    return f"`{term}` 系数为{direction}（coef={r['coef']:.4f}, p={r['p_value']:.3f}{r['significance']}），{sig}。"

fit_r1 = regression_fit_fixed[regression_fit_fixed["model"] == "R1_AR_only"]
fit_r2 = regression_fit_fixed[regression_fit_fixed["model"] == "R2_AR_news_lag1"]
fit_r3 = regression_fit_fixed[regression_fit_fixed["model"] == "R3_AR_news_lag12_macro"]
fit_sentence = ""
try:
    fit_sentence = f"R1 adjusted R²={float(fit_r1.iloc[0]['adj_r_squared']):.3f}；R2 adjusted R²={float(fit_r2.iloc[0]['adj_r_squared']):.3f}；R3 adjusted R²={float(fit_r3.iloc[0]['adj_r_squared']):.3f}。"
except Exception:
    fit_sentence = "部分模型未能提供可比较的 adjusted R²。"

reg_interp_fixed = f"""# 滞后回归修复后解释

修复后，核心新闻 lag1/lag2 变量已重新纳入回归：{describe_available(core_news_lag1_fixed + core_news_lag2_fixed)}。OLS 使用 HAC / Newey-West 稳健标准误，maxlags=4。

- {reg_term_sentence('weekly_layoff_news_count_lag1')}
- {reg_term_sentence('weekly_layoff_news_share_lag1')}
- 模型解释力：{fit_sentence}

若新闻变量系数为正且 p<0.10 或 p<0.05，可解释为存在滞后关联或前瞻信息；若不显著，则应如实说明证据不足。这里不做因果结论。
"""
safe_write_text(TEXT_DIR / "regression_interpretation_fixed.md", reg_interp_fixed)
print("Regression fixed core lag vars:", core_news_lag1_fixed + core_news_lag2_fixed)
display(regression_results_fixed.head(40))

# ---------- 7. RQ2 预测模型 fixed ----------
feature_sets_fixed = {
    "AR Baseline": ar_features_fixed,
    "Macro Only": ar_features_fixed + macro_features_fixed,
    "News Only": ar_features_fixed + news_features_fixed,
    "Macro + News": ar_features_fixed + macro_features_fixed + news_features_fixed,
}
feature_sets_fixed = {k: numeric_existing_features_fixed(df_fixed, v) for k, v in feature_sets_fixed.items()}

n_train_fixed = len(train_df_fixed)
n_splits_fixed = min(5, max(2, n_train_fixed // 12)) if n_train_fixed >= 24 else 2
tscv_fixed = TimeSeriesSplit(n_splits=n_splits_fixed)
alphas_fixed = np.logspace(-3, 3, 25)
models_fixed = {
    "Ridge": {"estimator": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", RidgeCV(alphas=alphas_fixed, cv=tscv_fixed))]), "target_transform": "log1p"},
    "Lasso": {"estimator": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", LassoCV(alphas=alphas_fixed, cv=tscv_fixed, random_state=RANDOM_STATE, max_iter=20000))]), "target_transform": "log1p"},
    "PoissonRegressor": {"estimator": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", PoissonRegressor(alpha=0.1, max_iter=2000))]), "target_transform": "raw"},
    "RandomForest": {"estimator": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(n_estimators=500, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1))]), "target_transform": "log1p"},
}
if XGBOOST_AVAILABLE:
    models_fixed["XGBoost"] = {"estimator": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=-1))]), "target_transform": "log1p"}

baseline_rows_fixed, pred_rows_fixed = [], []
for set_name, feats in feature_sets_fixed.items():
    feats = numeric_existing_features_fixed(df_fixed, feats)
    Xtr, ytr, used_feats, _ = prepare_xy_fixed(train_df_fixed, feats, main_target_fixed)
    Xte, yte, _, test_mask_y = prepare_xy_fixed(test_df_fixed, feats, main_target_fixed)
    used_feats = clean_feature_list_fixed(used_feats, Xte.columns)
    Xtr, Xte = Xtr[used_feats], Xte[used_feats]
    if not used_feats or len(Xtr) < 10 or len(Xte) < 2:
        baseline_rows_fixed.append({"feature_set": set_name, "model": "ALL", "status": f"skipped_no_features_or_small_sample_features={len(used_feats)}"})
        continue
    for model_name, spec in models_fixed.items():
        try:
            est = spec["estimator"]
            if spec["target_transform"] == "log1p":
                est.fit(Xtr, np.log1p(np.clip(ytr, 0, None)))
                pred = np.expm1(est.predict(Xte))
            else:
                est.fit(Xtr, np.clip(ytr, 0, None))
                pred = est.predict(Xte)
            pred = np.clip(pred, 0, None)
            metrics = evaluate_predictions_fixed(yte, pred)
            baseline_rows_fixed.append({"feature_set": set_name, "model": model_name, "status": "ok", "n_features": len(used_feats), "n_train": len(Xtr), "n_test": len(Xte), **metrics})
            for t, yt, yp in zip(test_df_fixed.loc[test_mask_y, time_col_fixed], yte, pred):
                pred_rows_fixed.append({"feature_set": set_name, "model": model_name, "time": t, "actual": yt, "predicted": yp})
        except Exception as e:
            baseline_rows_fixed.append({"feature_set": set_name, "model": model_name, "status": "failed", "error": str(e), "n_features": len(used_feats)})

baseline_model_results_fixed = pd.DataFrame(baseline_rows_fixed)
model_comparison_fixed = baseline_model_results_fixed[baseline_model_results_fixed["status"] == "ok"].sort_values(["MAE", "RMSE"]).reset_index(drop=True) if not baseline_model_results_fixed.empty else pd.DataFrame()
prediction_fixed = pd.DataFrame(pred_rows_fixed)
baseline_model_results_fixed.to_csv(RESULT_DIR / "baseline_model_results_fixed.csv", index=False, encoding="utf-8-sig")
model_comparison_fixed.to_csv(RESULT_DIR / "model_comparison_fixed.csv", index=False, encoding="utf-8-sig")
prediction_fixed.to_csv(RESULT_DIR / "prediction_records_fixed.csv", index=False, encoding="utf-8-sig")

if not model_comparison_fixed.empty:
    plot_df = model_comparison_fixed.copy()
    plot_df["label"] = plot_df["feature_set"] + "\n" + plot_df["model"]
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].barh(plot_df["label"], plot_df["MAE"])
    ax[0].set_title("Fixed MAE by model")
    ax[0].set_xlabel("MAE")
    ax[1].barh(plot_df["label"], plot_df["RMSE"])
    ax[1].set_title("Fixed RMSE by model")
    ax[1].set_xlabel("RMSE")
    plot_save_close(FIG_DIR / "fig4_model_comparison_mae_rmse_fixed.png")
    best_fixed = model_comparison_fixed.iloc[0]
    best_pred_fixed = prediction_fixed[(prediction_fixed["feature_set"] == best_fixed["feature_set"]) & (prediction_fixed["model"] == best_fixed["model"])]
    plt.figure(figsize=(10, 5))
    plt.plot(pd.to_datetime(best_pred_fixed["time"]), best_pred_fixed["actual"], marker="o", label="Actual")
    plt.plot(pd.to_datetime(best_pred_fixed["time"]), best_pred_fixed["predicted"], marker="o", label="Predicted")
    plt.title(f"Fixed prediction vs actual: {best_fixed['feature_set']} / {best_fixed['model']}")
    plt.xlabel("Time")
    plt.ylabel("Next-week layoff event count")
    plt.legend()
    plot_save_close(FIG_DIR / "prediction_vs_actual_best_model_fixed.png")
else:
    best_fixed = None

comparison_lines_fixed = []
if not model_comparison_fixed.empty:
    for m in sorted(model_comparison_fixed["model"].unique()):
        sub = model_comparison_fixed[model_comparison_fixed["model"] == m]
        mo = sub[sub["feature_set"] == "Macro Only"]
        mn = sub[sub["feature_set"] == "Macro + News"]
        if not mo.empty and not mn.empty:
            comparison_lines_fixed.append(f"- {m}: Macro + News 相比 Macro Only，MAE 变化 {mo.iloc[0]['MAE'] - mn.iloc[0]['MAE']:.3f}，RMSE 变化 {mo.iloc[0]['RMSE'] - mn.iloc[0]['RMSE']:.3f}。")
best_sentence_fixed = f"最佳组合为 `{best_fixed['feature_set']}` / `{best_fixed['model']}`，MAE={best_fixed['MAE']:.3f}，RMSE={best_fixed['RMSE']:.3f}。" if best_fixed is not None else "未得到有效预测结果。"
model_interp_fixed = "# 预测模型修复后解释\n\n" + best_sentence_fixed + "\n\n" + "\n".join(comparison_lines_fixed) + "\n\n如果提升只在部分模型或仅约 1% 左右，应表述为有限预测增量，而不是强结论。"
safe_write_text(TEXT_DIR / "model_comparison_interpretation_fixed.md", model_interp_fixed)
display(model_comparison_fixed.head(12))

# ---------- 8. 消融 fixed ----------
ablation_rows_fixed = []
if not model_comparison_fixed.empty:
    for m in sorted(model_comparison_fixed["model"].unique()):
        sub = model_comparison_fixed[model_comparison_fixed["model"] == m]
        mo = sub[sub["feature_set"] == "Macro Only"]
        mn = sub[sub["feature_set"] == "Macro + News"]
        if not mo.empty and not mn.empty:
            mo, mn = mo.iloc[0], mn.iloc[0]
            ablation_rows_fixed.append({"model": m, "MAE_macro_only": mo["MAE"], "MAE_macro_news": mn["MAE"], "RMSE_macro_only": mo["RMSE"], "RMSE_macro_news": mn["RMSE"], "improvement_mae_pct": (mo["MAE"] - mn["MAE"]) / mo["MAE"] * 100 if mo["MAE"] else np.nan, "improvement_rmse_pct": (mo["RMSE"] - mn["RMSE"]) / mo["RMSE"] * 100 if mo["RMSE"] else np.nan})
ablation_fixed = pd.DataFrame(ablation_rows_fixed)
ablation_fixed.to_csv(RESULT_DIR / "ablation_table_fixed.csv", index=False, encoding="utf-8-sig")
if not ablation_fixed.empty:
    x = np.arange(len(ablation_fixed)); w = 0.35
    plt.figure(figsize=(10, 5))
    plt.bar(x - w/2, ablation_fixed["improvement_mae_pct"], w, label="MAE improvement %")
    plt.bar(x + w/2, ablation_fixed["improvement_rmse_pct"], w, label="RMSE improvement %")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.xticks(x, ablation_fixed["model"], rotation=30, ha="right")
    plt.ylabel("Improvement (%)")
    plt.title("Fixed ablation: Macro Only vs Macro + News")
    plt.legend()
    plot_save_close(FIG_DIR / "ablation_macro_vs_macro_news_fixed.png")
    improved_models = ablation_fixed[(ablation_fixed["improvement_mae_pct"] > 0) | (ablation_fixed["improvement_rmse_pct"] > 0)]["model"].tolist()
    not_improved_models = ablation_fixed[(ablation_fixed["improvement_mae_pct"] <= 0) & (ablation_fixed["improvement_rmse_pct"] <= 0)]["model"].tolist()
    max_imp = np.nanmax([ablation_fixed["improvement_mae_pct"].max(), ablation_fixed["improvement_rmse_pct"].max()])
else:
    improved_models, not_improved_models, max_imp = [], [], np.nan
ablation_interp_fixed = f"""# 消融实验修复后解释

- 加入新闻变量后改善的模型：{describe_available(improved_models)}。
- 没有改善的模型：{describe_available(not_improved_models)}。
- 最大改善幅度约为 {max_imp:.2f}%（若为小幅提升，应谨慎表述）。

如果多数模型误差下降且幅度有实际意义，可以支持新闻变量具有预测增量；如果仅少数模型或小幅改善，则只能说预测增量有限。
"""
safe_write_text(TEXT_DIR / "ablation_interpretation_fixed.md", ablation_interp_fixed)
display(ablation_fixed)

# ---------- 9. 特征重要性 fixed ----------
ml_rows_fixed = []
macro_news_features_fixed = numeric_existing_features_fixed(df_fixed, feature_sets_fixed["Macro + News"])
Xtr, ytr, used_feats, _ = prepare_xy_fixed(train_df_fixed, macro_news_features_fixed, main_target_fixed)
Xte, yte, _, _ = prepare_xy_fixed(test_df_fixed, macro_news_features_fixed, main_target_fixed)
used_feats = clean_feature_list_fixed(used_feats, Xte.columns)
Xtr, Xte = Xtr[used_feats], Xte[used_feats]
importance_outputs_fixed = {}
if used_feats and len(Xtr) >= 10 and len(Xte) >= 2:
    rf_fixed = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(n_estimators=800, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1))])
    rf_fixed.fit(Xtr, np.log1p(np.clip(ytr, 0, None)))
    rf_pred = np.expm1(rf_fixed.predict(Xte))
    ml_rows_fixed.append({"model": "RandomForest", "status": "ok", "n_features": len(used_feats), **evaluate_predictions_fixed(yte, rf_pred)})
    rf_imp_fixed = pd.DataFrame({"feature": used_feats, "importance": rf_fixed.named_steps["model"].feature_importances_}).sort_values("importance", ascending=False).head(20)
    rf_imp_fixed.to_csv(RESULT_DIR / "feature_importance_random_forest_fixed.csv", index=False, encoding="utf-8-sig")
    importance_outputs_fixed["RandomForest"] = rf_imp_fixed
    plt.figure(figsize=(8, 6))
    plt.barh(rf_imp_fixed.sort_values("importance")["feature"], rf_imp_fixed.sort_values("importance")["importance"])
    plt.title("Fixed Random Forest feature importance")
    plt.xlabel("Importance")
    plot_save_close(FIG_DIR / "feature_importance_random_forest_fixed.png")
    if XGBOOST_AVAILABLE:
        try:
            xgb_fixed = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", XGBRegressor(n_estimators=400, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=-1))])
            xgb_fixed.fit(Xtr, np.log1p(np.clip(ytr, 0, None)))
            xgb_pred = np.expm1(xgb_fixed.predict(Xte))
            ml_rows_fixed.append({"model": "XGBoost", "status": "ok", "n_features": len(used_feats), **evaluate_predictions_fixed(yte, xgb_pred)})
            xgb_imp_fixed = pd.DataFrame({"feature": used_feats, "importance": xgb_fixed.named_steps["model"].feature_importances_}).sort_values("importance", ascending=False).head(20)
            xgb_imp_fixed.to_csv(RESULT_DIR / "feature_importance_xgboost_fixed.csv", index=False, encoding="utf-8-sig")
            importance_outputs_fixed["XGBoost"] = xgb_imp_fixed
            plt.figure(figsize=(8, 6))
            plt.barh(xgb_imp_fixed.sort_values("importance")["feature"], xgb_imp_fixed.sort_values("importance")["importance"])
            plt.title("Fixed XGBoost feature importance")
            plt.xlabel("Importance")
            plot_save_close(FIG_DIR / "feature_importance_xgboost_fixed.png")
        except Exception as e:
            ml_rows_fixed.append({"model": "XGBoost", "status": "failed", "error": str(e)})
    else:
        ml_rows_fixed.append({"model": "XGBoost", "status": "skipped_not_installed", "error": globals().get("XGBOOST_IMPORT_ERROR", "not installed")})
else:
    ml_rows_fixed.append({"model": "FeatureImportance", "status": "skipped_no_features_or_small_sample"})
    pd.DataFrame(columns=["feature", "importance"]).to_csv(RESULT_DIR / "feature_importance_random_forest_fixed.csv", index=False, encoding="utf-8-sig")
ml_results_fixed = pd.DataFrame(ml_rows_fixed)
ml_results_fixed.to_csv(RESULT_DIR / "ml_model_results_fixed.csv", index=False, encoding="utf-8-sig")

fi_lines = []
for model_name, imp in importance_outputs_fixed.items():
    top = imp["feature"].tolist()
    watched = [f for f in ["weekly_layoff_news_count_lag1", "weekly_layoff_news_count_lag2", "weekly_layoff_news_share_lag1", "weekly_layoff_news_share_lag2"] if f in top]
    ar_top = [f for f in top if f in ar_features_fixed]
    macro_top = [f for f in top if f in macro_features_fixed]
    fi_lines.append(f"- {model_name}: 关注新闻 lag 进入前20的变量为 {describe_available(watched)}；AR 前20变量 {len(ar_top)} 个；宏观前20变量 {len(macro_top)} 个。")
if not fi_lines:
    fi_lines.append("- 未能生成有效特征重要性。")
safe_write_text(TEXT_DIR / "feature_importance_interpretation_fixed.md", "# 特征重要性修复后解释\n\n" + "\n".join(fi_lines))
display(ml_results_fixed)

# ---------- 10. 稳健性 fixed ----------
def run_compare_fixed(target_col, feats_a, feats_b, label, models_to_run=("Ridge", "RandomForest")):
    rows = []
    if target_col not in df_fixed.columns:
        return [{"robustness_check": label, "status": f"skipped_missing_target_{target_col}"}]
    for set_name, feats in [("Macro Only", feats_a), ("Macro + News", feats_b)]:
        feats = numeric_existing_features_fixed(df_fixed, feats)
        Xtr, ytr, used, _ = prepare_xy_fixed(train_df_fixed, feats, target_col)
        Xte, yte, _, _ = prepare_xy_fixed(test_df_fixed, feats, target_col)
        used = clean_feature_list_fixed(used, Xte.columns)
        Xtr, Xte = Xtr[used], Xte[used]
        if not used or len(Xtr) < 10 or len(Xte) < 2:
            rows.append({"robustness_check": label, "feature_set": set_name, "status": "skipped_no_features_or_small_sample"})
            continue
        for m in models_to_run:
            try:
                est = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]) if m == "Ridge" else Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1))])
                est.fit(Xtr, np.log1p(np.clip(ytr, 0, None)))
                pred = np.expm1(est.predict(Xte))
                rows.append({"robustness_check": label, "target": target_col, "feature_set": set_name, "model": m, "status": "ok", **evaluate_predictions_fixed(yte, pred)})
            except Exception as e:
                rows.append({"robustness_check": label, "feature_set": set_name, "model": m, "status": "failed", "error": str(e)})
    return rows

robust_rows_fixed = []
alt_target_fixed = next((t for t in ["target_next_week_known_layoff_count", "target_next_week_log_known_layoff_count"] if t in df_fixed.columns), None)
if alt_target_fixed:
    robust_rows_fixed.extend(run_compare_fixed(alt_target_fixed, ar_features_fixed + macro_features_fixed, ar_features_fixed + macro_features_fixed + news_features_fixed, "alternative_target_known_layoff"))
else:
    robust_rows_fixed.append({"robustness_check": "alternative_target_known_layoff", "status": "skipped_no_known_layoff_target"})
layoff_only_news_fixed = []
for b in ["weekly_layoff_news_count", "weekly_layoff_news_share", "weekly_layoff_avg_sentiment"]:
    layoff_only_news_fixed.append(b)
    for k in range(1, 5): layoff_only_news_fixed.append(f"{b}_lag{k}")
layoff_only_news_fixed = numeric_existing_features_fixed(df_fixed, layoff_only_news_fixed)
if layoff_only_news_fixed:
    robust_rows_fixed.extend(run_compare_fixed(main_target_fixed, ar_features_fixed + macro_features_fixed, ar_features_fixed + macro_features_fixed + layoff_only_news_fixed, "layoff_only_news_features"))
else:
    robust_rows_fixed.append({"robustness_check": "layoff_only_news_features", "status": "skipped_no_layoff_only_news_features"})
for label, lags in [("news_lag1_only", [1]), ("news_lag1_lag2", [1, 2]), ("news_lag1_to_lag4", [1, 2, 3, 4])]:
    lag_feats = [c for c in news_features_fixed if any(c.lower().endswith(f"_lag{k}") for k in lags)]
    if lag_feats:
        robust_rows_fixed.extend(run_compare_fixed(main_target_fixed, ar_features_fixed + macro_features_fixed, ar_features_fixed + macro_features_fixed + lag_feats, label))
    else:
        robust_rows_fixed.append({"robustness_check": label, "status": "skipped_no_matching_lag_features"})
robustness_fixed = pd.DataFrame(robust_rows_fixed)
robustness_fixed.to_csv(RESULT_DIR / "robustness_results_fixed.csv", index=False, encoding="utf-8-sig")
robust_lines_fixed = []
okrob = robustness_fixed[robustness_fixed.get("status", pd.Series(dtype=str)) == "ok"] if not robustness_fixed.empty else pd.DataFrame()
if not okrob.empty:
    for check in okrob["robustness_check"].dropna().unique():
        sub = okrob[okrob["robustness_check"] == check]
        total = improved = 0
        for m in sub["model"].dropna().unique():
            mo = sub[(sub["model"] == m) & (sub["feature_set"] == "Macro Only")]
            mn = sub[(sub["model"] == m) & (sub["feature_set"] == "Macro + News")]
            if not mo.empty and not mn.empty:
                total += 1
                if mn.iloc[0]["MAE"] < mo.iloc[0]["MAE"] or mn.iloc[0]["RMSE"] < mo.iloc[0]["RMSE"]:
                    improved += 1
        robust_lines_fixed.append(f"- {check}: {improved}/{total} 个模型在 MAE 或 RMSE 上改善。")
else:
    robust_lines_fixed.append("- 没有有效稳健性结果。")
safe_write_text(TEXT_DIR / "robustness_interpretation_fixed.md", "# 稳健性检验修复后解释\n\n" + "\n".join(robust_lines_fixed))
display(robustness_fixed.head(30))

# ---------- 11. AI vs Non-AI fixed ----------
hetero_rows_fixed, hetero_tlcc_rows_fixed = [], []
ai_col_fixed = "weekly_ai_layoff_event_count"
nonai_col_fixed = "weekly_non_ai_layoff_event_count"
if ai_col_fixed in df_fixed.columns and nonai_col_fixed in df_fixed.columns:
    plt.figure(figsize=(11, 5))
    plt.plot(df_fixed[time_col_fixed], pd.to_numeric(df_fixed[ai_col_fixed], errors="coerce"), label="AI layoffs", marker="o", markersize=3)
    plt.plot(df_fixed[time_col_fixed], pd.to_numeric(df_fixed[nonai_col_fixed], errors="coerce"), label="Non-AI layoffs", marker="o", markersize=3)
    plt.title("Fixed AI vs Non-AI weekly layoff event counts")
    plt.xlabel("Time")
    plt.ylabel("Event count")
    plt.legend()
    plot_save_close(FIG_DIR / "fig5_ai_vs_nonai_trend_fixed.png")
    hetero_news_vars_fixed = [v for v in ["weekly_layoff_news_count", "weekly_layoff_news_share", "weekly_negative_share", "sentiment_shock"] if v in df_fixed.columns]
    for group, out_col in [("AI", ai_col_fixed), ("Non-AI", nonai_col_fixed)]:
        y0 = pd.to_numeric(df_fixed[out_col], errors="coerce")
        for var in hetero_news_vars_fixed:
            x0 = pd.to_numeric(df_fixed[var], errors="coerce")
            for lag in [1, 2]:
                pair = pd.DataFrame({"x": x0.shift(lag), "y": y0}).dropna()
                if len(pair) >= 3 and pair["x"].std() > 0 and pair["y"].std() > 0:
                    corr, pval = stats.pearsonr(pair["x"], pair["y"])
                else:
                    corr, pval = np.nan, np.nan
                hetero_tlcc_rows_fixed.append({"group": group, "variable": var, "lag": lag, "correlation": corr, "p_value": pval, "n_obs": len(pair), "significance": sig_star_fixed(pval)})
    ai_target_fixed = "target_next_week_ai_layoff_event_count"
    nonai_target_fixed = "target_next_week_non_ai_layoff_event_count"
    df_fixed[ai_target_fixed] = pd.to_numeric(df_fixed[ai_col_fixed], errors="coerce").shift(-1)
    df_fixed[nonai_target_fixed] = pd.to_numeric(df_fixed[nonai_col_fixed], errors="coerce").shift(-1)
    df_fixed["target_next_week_log_ai_layoff_event_count"] = np.log1p(df_fixed[ai_target_fixed])
    df_fixed["target_next_week_log_non_ai_layoff_event_count"] = np.log1p(df_fixed[nonai_target_fixed])
    h_feats = numeric_existing_features_fixed(df_fixed, ["weekly_layoff_news_count_lag1", "weekly_layoff_news_count_lag2", "weekly_layoff_news_share_lag1", "weekly_layoff_news_share_lag2", "weekly_negative_share_lag1", "weekly_negative_share_lag2", "sentiment_shock_lag1", "sentiment_shock_lag2"] + ar_controls_fixed)
    for group, ycol in [("AI", "target_next_week_log_ai_layoff_event_count"), ("Non-AI", "target_next_week_log_non_ai_layoff_event_count")]:
        hdf = df_fixed[[ycol] + h_feats].copy()
        for c in [ycol] + h_feats: hdf[c] = pd.to_numeric(hdf[c], errors="coerce")
        hdf = hdf.dropna()
        if h_feats and len(hdf) >= max(10, len(h_feats)+5):
            try:
                hf = sm.OLS(hdf[ycol], sm.add_constant(hdf[h_feats], has_constant="add")).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
                for term in hf.params.index:
                    hetero_rows_fixed.append({"group": group, "model": "heterogeneity_OLS_HAC", "status": "ok", "term": term, "coef": hf.params[term], "std_err": hf.bse[term], "stat": hf.tvalues[term], "p_value": hf.pvalues[term], "significance": sig_star_fixed(hf.pvalues[term]), "n_obs": int(hf.nobs), "r_squared": hf.rsquared})
            except Exception as e:
                hetero_rows_fixed.append({"group": group, "status": "failed", "error": str(e)})
        else:
            hetero_rows_fixed.append({"group": group, "status": "skipped_small_sample_or_no_features"})
else:
    hetero_rows_fixed.append({"group": "AI_vs_NonAI", "status": "skipped_missing_ai_nonai_columns"})
hetero_tlcc_fixed = pd.DataFrame(hetero_tlcc_rows_fixed)
hetero_results_fixed = pd.DataFrame(hetero_rows_fixed)
hetero_tlcc_fixed.to_csv(RESULT_DIR / "heterogeneity_tlcc_ai_nonai_fixed.csv", index=False, encoding="utf-8-sig")
hetero_results_fixed.to_csv(RESULT_DIR / "heterogeneity_results_fixed.csv", index=False, encoding="utf-8-sig")
hetero_interp_fixed = "# AI vs 非 AI 异质性修复后解释\n\n本部分优先使用 weekly_layoff_news_count 和 weekly_layoff_news_share 的 lag1/lag2 检查 AI 与非 AI 裁员事件的滞后关联。结果只用于描述性比较，不作因果解释。"
if not hetero_tlcc_fixed.empty:
    for group in ["AI", "Non-AI"]:
        sub = hetero_tlcc_fixed[hetero_tlcc_fixed["group"] == group].dropna(subset=["correlation"]).copy()
        if not sub.empty:
            sub["abs_corr"] = sub["correlation"].abs()
            r = sub.sort_values("abs_corr", ascending=False).iloc[0]
            hetero_interp_fixed += f"\n\n- {group}: 最强 lag1/lag2 相关为 `{r['variable']}` lag{int(r['lag'])}，corr={r['correlation']:.3f}, p={r['p_value']:.3f}{r['significance']}。"
safe_write_text(TEXT_DIR / "heterogeneity_interpretation_fixed.md", hetero_interp_fixed)
display(hetero_tlcc_fixed.head(20))

# ---------- 12. 修复后质量检查 ----------
all_model_features_fixed = clean_feature_list_fixed(ar_features_fixed + macro_features_fixed + news_features_fixed + heterogeneity_features_fixed)
quality_rows_fixed = []
def qc(name, passed, detail):
    quality_rows_fixed.append({"check": name, "passed": bool(passed), "detail": detail})

bad_future = [f for f in all_model_features_fixed if any(k in f.lower() for k in ["target_next_week", "next_week", "future", "lead", "t_plus"])]
qc("模型特征不包含 target_next_week / future 信息", len(bad_future) == 0, ", ".join(bad_future) if bad_future else "OK")
weekly_core_bad = [c for c in core_should_include if c in df_fixed.columns and is_forbidden_feature_fixed(c)]
qc("weekly_ 核心变量不再因 y_ 被误删", len(weekly_core_bad) == 0, ", ".join(weekly_core_bad) if weekly_core_bad else "OK")
qc("news_features 不只剩 sentiment_shock", len(news_features_fixed) > 6 and any("layoff_news" in f for f in news_features_fixed), f"news_features={len(news_features_fixed)}")
raw_ar_exists = any("layoff_event_count_lag" in c.lower() or "layoff_event_count_roll" in c.lower() for c in df_fixed.columns)
qc("ar_features 非空，除非原始数据没有 AR 变量", len(ar_features_fixed) > 0 or not raw_ar_exists, f"ar_features={len(ar_features_fixed)}, raw_ar_exists={raw_ar_exists}")
qc("TLCC 包含 weekly_layoff_news_count 或 weekly_layoff_news_share", any(v in tlcc_vars_fixed for v in ["weekly_layoff_news_count", "weekly_layoff_news_share"]), str(tlcc_vars_fixed))
reg_terms = regression_results_fixed["term"].tolist() if not regression_results_fixed.empty else []
qc("回归包含 layoff_news lag1 或 lag2", any(t in reg_terms for t in ["weekly_layoff_news_count_lag1", "weekly_layoff_news_count_lag2", "weekly_layoff_news_share_lag1", "weekly_layoff_news_share_lag2"]), ", ".join([t for t in reg_terms if "layoff_news" in t]))
qc("Macro + News 特征数大于 Macro Only", len(feature_sets_fixed["Macro + News"]) > len(feature_sets_fixed["Macro Only"]), f"Macro+News={len(feature_sets_fixed['Macro + News'])}, Macro Only={len(feature_sets_fixed['Macro Only'])}")
qc("target_next_week_* 不在任何模型特征中", not any(f.startswith("target_next_week") for f in all_model_features_fixed), "OK" if not any(f.startswith("target_next_week") for f in all_model_features_fixed) else "FAILED")

quality_fixed = pd.DataFrame(quality_rows_fixed)
quality_fixed.to_csv(RESULT_DIR / "memberB_quality_check_after_fix.csv", index=False, encoding="utf-8-sig")

# 修复摘要。
macro_news_comp_sentence = ""
if not ablation_fixed.empty:
    improved_count = int(((ablation_fixed["improvement_mae_pct"] > 0) | (ablation_fixed["improvement_rmse_pct"] > 0)).sum())
    macro_news_comp_sentence = f"Macro + News 与 Macro Only 已重新公平比较，{improved_count}/{len(ablation_fixed)} 个模型在 MAE 或 RMSE 上有改善。"
else:
    macro_news_comp_sentence = "Macro + News 与 Macro Only 的消融比较未得到有效成对结果。"

fix_summary = f"""# 成员B Notebook 修复摘要

## 1. 原始问题

旧版 `is_forbidden_feature` 使用 `"y_" in col` 判断标签列，导致 `weekly_` 开头的真实解释变量被误删。例如 `weekly_layoff_news_count`、`weekly_layoff_news_share`、`weekly_negative_share` 和 `weekly_layoff_event_count_lag1` 都可能被错误排除。

## 2. 修复内容

已改为只禁止真正名为 `y` 或以 `y_` 开头的标签列，同时继续禁止 `target_next_week`、`next_week`、`future`、`lead`、`t_plus`、`label` 等未来信息列。修复后允许 `weekly_`、`*_lag1` 到 `*_lag4`、rolling 历史变量进入模型。

## 3. 修复后核心变量纳入情况

- news_features: {len(news_features_fixed)} 个变量，包括 {describe_available([f for f in news_features_fixed if 'layoff_news' in f or 'negative_share' in f][:12])}
- ar_features: {len(ar_features_fixed)} 个变量，包括 {describe_available(ar_features_fixed)}
- macro_features: {len(macro_features_fixed)} 个变量，包括 {describe_available(macro_features_fixed)}
- heterogeneity_features: {len(heterogeneity_features_fixed)} 个变量。

## 4. 修复后 RQ1 结果

TLCC 已包含 `weekly_layoff_news_count` 与 `weekly_layoff_news_share`。{tlcc_lag12_sentence('weekly_layoff_news_count')} {tlcc_lag12_sentence('weekly_layoff_news_share')} 回归已包含 layoff_news 的 lag1/lag2 变量；{reg_term_sentence('weekly_layoff_news_count_lag1')} {reg_term_sentence('weekly_layoff_news_share_lag1')}

## 5. 修复后 RQ2 结果

{macro_news_comp_sentence} 若改善幅度较小，应写为有限预测增量，不能夸大。

## 6. 是否可以用于报告

修复后的 `_fixed` 结果已经通过自动质量检查，且关键 proposal 变量被重新纳入。可以优先使用 `_fixed` 文件写报告。

## 7. 限制

结果仍受周度样本量、新闻噪声、变量共线性、裁员事件突发性影响。所有结果最多支持“滞后关联”或“前瞻信息”表述，不应写成因果证明。
"""
safe_write_text(TEXT_DIR / "memberB_fix_summary.md", fix_summary)

print("修复后质量检查：")
display(quality_fixed)
print(fix_summary)


开始执行修复后完整重跑流程...
修复后变量组数量：
news_features_fixed = 53
macro_features_fixed = 12
ar_features_fixed = 5
heterogeneity_features_fixed = 4
forbidden_features_fixed = 4
核心变量诊断：
                                  column  exists_in_data assigned_group  is_forbidden  should_be_included              note
0               weekly_layoff_news_count            True  news_features         False                True  OK: 核心变量已纳入对应变量组
1          weekly_layoff_news_count_lag1            True  news_features         False                True  OK: 核心变量已纳入对应变量组
2          weekly_layoff_news_count_lag2            True  news_features         False                True  OK: 核心变量已纳入对应变量组
3               weekly_layoff_news_share            True  news_features         False                True  OK: 核心变量已纳入对应变量组
4          weekly_layoff_news_share_lag1            True  news_features         False                True  OK: 核心变量已纳入对应变量组
5          weekly_layoff_news_share_lag2            True  news_features         False 


## Prediction Improvement：新增预测改进

本章节在不删除原有结果的基础上，针对修复后 `Macro + News` 预测表现差的问题进行改进。核心思路是减少新闻特征噪声、只在训练集内做特征筛选、尝试两阶段模型和更适合计数目标的模型，并用 expanding-window rolling validation 检查结论稳定性。所有步骤均不随机打乱时间序列，不允许 `target_next_week_*` 或未来信息进入特征，也不删除零裁员周。


In [16]:

# ==============================
# PREDICTION IMPROVEMENT SECTION
# 新增预测改进：特征筛选、两阶段模型、计数模型、滚动验证与误差诊断
# ==============================

print("开始 Prediction Improvement 章节...")

from sklearn.linear_model import LogisticRegression, TweedieRegressor
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.inspection import permutation_importance

try:
    from xgboost import XGBClassifier
    XGB_CLASSIFIER_AVAILABLE = True
except Exception as e:
    XGBClassifier = None
    XGB_CLASSIFIER_AVAILABLE = False
    XGB_CLASSIFIER_IMPORT_ERROR = str(e)

# ---------- A. 安全工具函数 ----------
def pi_forbidden(col):
    """Prediction Improvement 专用防泄漏检查。"""
    low = str(col).lower()
    if low == "y" or low.startswith("y_"):
        return True
    return any(k in low for k in ["target_next_week", "next_week", "future", "lead", "t_plus", "label"])

def pi_clean_features(features, data):
    """保序去重，只保留存在、数值可用、非未来信息的特征。"""
    seen, out = set(), []
    for f in features:
        if f in seen or f not in data.columns or pi_forbidden(f):
            continue
        s = pd.to_numeric(data[f], errors="coerce")
        if s.notna().sum() > 0:
            seen.add(f)
            out.append(f)
    return out

def pi_prepare_xy(data, features, target):
    feats = pi_clean_features(features, data)
    X = data[feats].copy()
    for c in feats:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    y = pd.to_numeric(data[target], errors="coerce")
    mask = y.notna()
    return X.loc[mask], y.loc[mask], feats, mask

def pi_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    denom = np.abs(y_true) + np.abs(y_pred)
    smape_val = np.mean(2*np.abs(y_pred[denom>0]-y_true[denom>0])/denom[denom>0]) if (denom>0).sum() else np.nan
    da = np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred))) if len(y_true) >= 2 else np.nan
    try:
        pdev = mean_poisson_deviance(np.clip(y_true, 0, None), np.clip(y_pred, 1e-9, None))
    except Exception:
        pdev = np.nan
    nz = y_true > 0
    mape_nonzero = mean_absolute_percentage_error(y_true[nz], y_pred[nz]) if nz.sum() else np.nan
    return {"MAE": mae, "RMSE": rmse, "MAPE_nonzero": mape_nonzero, "sMAPE": smape_val, "Poisson_Deviance": pdev, "Directional_Accuracy": da}

def pi_fit_predict_reg(model_name, X_train, y_train, X_test, transform="log1p"):
    """统一训练回归模型并返回非负预测。"""
    if model_name == "Ridge":
        est = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))])
    elif model_name == "Lasso":
        est = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", Lasso(alpha=0.01, max_iter=20000, random_state=RANDOM_STATE))])
    elif model_name == "PoissonRegressor":
        est = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", PoissonRegressor(alpha=0.1, max_iter=2000))])
        transform = "raw"
    elif model_name == "RandomForest":
        est = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(n_estimators=500, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1))])
    elif model_name == "HistGBR_Poisson":
        est = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", HistGradientBoostingRegressor(loss="poisson", max_iter=200, learning_rate=0.05, min_samples_leaf=10, random_state=RANDOM_STATE))])
        transform = "raw"
    elif model_name.startswith("Tweedie"):
        power = float(model_name.split("_")[-1])
        est = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", TweedieRegressor(power=power, alpha=0.1, link="log", max_iter=2000))])
        transform = "raw"
    elif model_name == "XGB_count_poisson" and XGBOOST_AVAILABLE:
        est = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, objective="count:poisson", random_state=RANDOM_STATE, n_jobs=-1))])
        transform = "raw"
    else:
        raise ValueError(f"未知或不可用模型：{model_name}")
    if transform == "log1p":
        est.fit(X_train, np.log1p(np.clip(y_train, 0, None)))
        pred = np.expm1(est.predict(X_test))
    else:
        est.fit(X_train, np.clip(y_train, 0, None))
        pred = est.predict(X_test)
    return np.clip(pred, 0, None), est

# ---------- B. 新增 log 新闻数量特征，不使用未来信息 ----------
log_base_vars = ["weekly_news_count", "weekly_layoff_news_count", "weekly_hiring_news_count"]
log_created = []
for base in log_base_vars:
    for suffix in [""] + [f"_lag{k}" for k in range(1, 5)]:
        src = f"{base}{suffix}"
        dst = f"log1p_{src}"
        if src in df_fixed.columns:
            df_fixed[dst] = np.log1p(np.clip(pd.to_numeric(df_fixed[src], errors="coerce"), 0, None))
            train_df_fixed[dst] = np.log1p(np.clip(pd.to_numeric(train_df_fixed[src], errors="coerce"), 0, None)) if src in train_df_fixed.columns else np.nan
            test_df_fixed[dst] = np.log1p(np.clip(pd.to_numeric(test_df_fixed[src], errors="coerce"), 0, None)) if src in test_df_fixed.columns else np.nan
            log_created.append(dst)
print("新增 log1p 新闻数量特征：", log_created)

# ---------- C. 核心新闻与训练集内特征筛选 ----------
core_news_candidates = [
    "weekly_layoff_news_count_lag1",
    "weekly_layoff_news_count_lag2",
    "weekly_layoff_news_share_lag1",
    "weekly_layoff_news_share_lag2",
    "weekly_negative_share_lag1",
    "weekly_negative_share_lag2",
    "sentiment_shock_lag1",
    "sentiment_shock_lag2",
    "weekly_layoff_avg_sentiment_lag1",
    "weekly_layoff_avg_sentiment_lag2",
]
core_news_features_pi = pi_clean_features(core_news_candidates, df_fixed)
missing_core_news_pi = [f for f in core_news_candidates if f not in core_news_features_pi]

# news pool 包含修复后的 news_features 与新增 log 特征；筛选只用训练集。
news_pool_pi = pi_clean_features(news_features_fixed + log_created, df_fixed)
macro_only_pi = pi_clean_features(ar_features_fixed + macro_features_fixed, df_fixed)
full_news_pi = pi_clean_features(news_features_fixed + log_created, df_fixed)
macro_full_news_pi = pi_clean_features(ar_features_fixed + macro_features_fixed + full_news_pi, df_fixed)

# 训练目标。
y_train_pi = pd.to_numeric(train_df_fixed[main_target_fixed], errors="coerce")

# correlation top 10：仅训练集计算相关。
corr_rows_pi = []
for f in news_pool_pi:
    pair = pd.DataFrame({"x": pd.to_numeric(train_df_fixed[f], errors="coerce"), "y": y_train_pi}).dropna()
    if len(pair) >= 5 and pair["x"].std() > 0 and pair["y"].std() > 0:
        corr = pair["x"].corr(pair["y"])
        corr_rows_pi.append({"feature": f, "corr": corr, "abs_corr": abs(corr)})
corr_select_df_pi = pd.DataFrame(corr_rows_pi).sort_values("abs_corr", ascending=False)
corr_top_news_pi = corr_select_df_pi.head(10)["feature"].tolist() if not corr_select_df_pi.empty else []

# LassoCV selected：仅训练集 fit。
X_train_news_pi, y_train_news_pi, used_news_pi, _ = pi_prepare_xy(train_df_fixed, news_pool_pi, main_target_fixed)
lasso_selected_news_pi = []
if len(used_news_pi) > 0 and len(X_train_news_pi) >= 20:
    try:
        lasso_sel_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LassoCV(alphas=np.logspace(-4, 1, 40), cv=TimeSeriesSplit(n_splits=min(5, max(2, len(X_train_news_pi)//20))), random_state=RANDOM_STATE, max_iter=30000)),
        ])
        lasso_sel_pipe.fit(X_train_news_pi, np.log1p(np.clip(y_train_news_pi, 0, None)))
        coefs = lasso_sel_pipe.named_steps["model"].coef_
        lasso_selected_news_pi = [f for f, coef in zip(used_news_pi, coefs) if abs(coef) > 1e-8]
        # 防止 Lasso 过度稀疏，最多补充训练相关性前 5 个。
        if not lasso_selected_news_pi:
            lasso_selected_news_pi = corr_top_news_pi[:5]
    except Exception as e:
        lasso_selected_news_pi = corr_top_news_pi[:5]
        print("LassoCV 特征筛选失败，改用 corr top5：", e)

# RF top 10：仅训练集 fit。
rf_top_news_pi = []
if len(used_news_pi) > 0 and len(X_train_news_pi) >= 20:
    try:
        rf_sel_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(n_estimators=500, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)),
        ])
        rf_sel_pipe.fit(X_train_news_pi, np.log1p(np.clip(y_train_news_pi, 0, None)))
        imp = rf_sel_pipe.named_steps["model"].feature_importances_
        rf_top_news_pi = pd.DataFrame({"feature": used_news_pi, "importance": imp}).sort_values("importance", ascending=False).head(10)["feature"].tolist()
    except Exception as e:
        rf_top_news_pi = corr_top_news_pi[:10]
        print("RF 特征筛选失败，改用 corr top10：", e)

selected_news_union_pi = pi_clean_features(core_news_features_pi + corr_top_news_pi + lasso_selected_news_pi + rf_top_news_pi, df_fixed)

selection_summary_pi = pd.DataFrame([
    {"selection_method": "Core News", "n_features": len(core_news_features_pi), "features": ";".join(core_news_features_pi), "note": "手工指定的 proposal 核心新闻 lag1/lag2 变量"},
    {"selection_method": "CorrTop News", "n_features": len(corr_top_news_pi), "features": ";".join(corr_top_news_pi), "note": "仅训练集上与目标相关性绝对值 top10"},
    {"selection_method": "LassoSelected News", "n_features": len(lasso_selected_news_pi), "features": ";".join(lasso_selected_news_pi), "note": "仅训练集 LassoCV 非零系数"},
    {"selection_method": "RFTop News", "n_features": len(rf_top_news_pi), "features": ";".join(rf_top_news_pi), "note": "仅训练集 RandomForest 重要性 top10"},
    {"selection_method": "Selected News Union", "n_features": len(selected_news_union_pi), "features": ";".join(selected_news_union_pi), "note": "用于 Macro + Selected News"},
])
selection_summary_pi.to_csv(RESULT_DIR / "prediction_improvement_feature_selection.csv", index=False, encoding="utf-8-sig")
print("缺失的核心新闻变量：", missing_core_news_pi)
display(selection_summary_pi)

# ---------- D. Model 4 与筛选特征组公平比较 ----------
feature_groups_pi = {
    "Macro Only": macro_only_pi,
    "Macro + Full News": macro_full_news_pi,
    "Macro + Core News": pi_clean_features(ar_features_fixed + macro_features_fixed + core_news_features_pi, df_fixed),
    "Macro + CorrTop News": pi_clean_features(ar_features_fixed + macro_features_fixed + corr_top_news_pi, df_fixed),
    "Macro + LassoSelected News": pi_clean_features(ar_features_fixed + macro_features_fixed + lasso_selected_news_pi, df_fixed),
    "Macro + RFTop News": pi_clean_features(ar_features_fixed + macro_features_fixed + rf_top_news_pi, df_fixed),
    "Macro + Selected News": pi_clean_features(ar_features_fixed + macro_features_fixed + selected_news_union_pi, df_fixed),
}

base_model_names_pi = ["Ridge", "Lasso", "PoissonRegressor", "RandomForest"]
comparison_rows_pi = []
prediction_rows_pi = []
for group_name, feats in feature_groups_pi.items():
    Xtr, ytr, used, _ = pi_prepare_xy(train_df_fixed, feats, main_target_fixed)
    Xte, yte, _, test_mask = pi_prepare_xy(test_df_fixed, feats, main_target_fixed)
    used = [f for f in used if f in Xte.columns]
    Xtr, Xte = Xtr[used], Xte[used]
    if not used or len(Xtr) < 20 or len(Xte) < 2:
        comparison_rows_pi.append({"model_group": group_name, "model_name": "ALL", "feature_set": ";".join(used), "status": "skipped_no_features_or_small_sample"})
        continue
    for model_name in base_model_names_pi:
        try:
            pred, est = pi_fit_predict_reg(model_name, Xtr, ytr, Xte)
            metrics = pi_metrics(yte, pred)
            comparison_rows_pi.append({"model_group": group_name, "model_name": model_name, "feature_set": group_name, "status": "ok", "n_features": len(used), **metrics})
            for t, yt, yp in zip(test_df_fixed.loc[test_mask, time_col_fixed], yte, pred):
                prediction_rows_pi.append({"model_group": group_name, "model_name": model_name, "week": t, "y_true": yt, "y_pred": yp})
        except Exception as e:
            comparison_rows_pi.append({"model_group": group_name, "model_name": model_name, "feature_set": group_name, "status": "failed", "error": str(e), "n_features": len(used)})

# ---------- E. 更适合计数目标的模型 ----------
count_model_rows_pi = []
count_model_preds_pi = []
count_models_pi = ["PoissonRegressor"] + [f"Tweedie_{p}" for p in [1.1, 1.3, 1.5, 1.7, 1.9]] + ["HistGBR_Poisson"]
if XGBOOST_AVAILABLE:
    count_models_pi.append("XGB_count_poisson")
count_feature_group_pi = "Macro + Core News"
count_feats_pi = feature_groups_pi[count_feature_group_pi]
Xtr_c, ytr_c, used_c, _ = pi_prepare_xy(train_df_fixed, count_feats_pi, main_target_fixed)
Xte_c, yte_c, _, test_mask_c = pi_prepare_xy(test_df_fixed, count_feats_pi, main_target_fixed)
used_c = [f for f in used_c if f in Xte_c.columns]
Xtr_c, Xte_c = Xtr_c[used_c], Xte_c[used_c]
for model_name in count_models_pi:
    try:
        pred, est = pi_fit_predict_reg(model_name, Xtr_c, ytr_c, Xte_c, transform="raw")
        metrics = pi_metrics(yte_c, pred)
        row = {"model_group": "Count Models", "model_name": model_name, "feature_set": count_feature_group_pi, "status": "ok", "n_features": len(used_c), **metrics}
        count_model_rows_pi.append(row)
        comparison_rows_pi.append(row.copy())
        for t, yt, yp in zip(test_df_fixed.loc[test_mask_c, time_col_fixed], yte_c, pred):
            count_model_preds_pi.append({"model_group": "Count Models", "model_name": model_name, "week": t, "y_true": yt, "y_pred": yp})
            prediction_rows_pi.append({"model_group": "Count Models", "model_name": model_name, "week": t, "y_true": yt, "y_pred": yp})
    except Exception as e:
        row = {"model_group": "Count Models", "model_name": model_name, "feature_set": count_feature_group_pi, "status": "failed", "error": str(e), "n_features": len(used_c)}
        count_model_rows_pi.append(row)
        comparison_rows_pi.append(row.copy())

count_model_results_pi = pd.DataFrame(count_model_rows_pi)
count_model_results_pi.to_csv(RESULT_DIR / "count_model_results.csv", index=False, encoding="utf-8-sig")

# ---------- F. 两阶段模型 ----------
def pi_fit_classifier(name, X_train, y_binary, X_test):
    if name == "LogisticRegression":
        clf = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))])
    elif name == "RandomForestClassifier":
        clf = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestClassifier(n_estimators=500, min_samples_leaf=3, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1))])
    elif name == "XGBoostClassifier" and XGB_CLASSIFIER_AVAILABLE:
        clf = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1))])
    else:
        raise ValueError(f"分类模型不可用：{name}")
    clf.fit(X_train, y_binary)
    pred_label = clf.predict(X_test)
    if hasattr(clf, "predict_proba"):
        proba = clf.predict_proba(X_test)[:, 1]
    else:
        try:
            proba = clf.named_steps["model"].predict_proba(clf.named_steps["imputer"].transform(X_test))[:, 1]
        except Exception:
            proba = pred_label.astype(float)
    return pred_label, proba, clf

two_stage_rows_pi = []
two_stage_pred_rows_pi = []
two_stage_feats_pi = feature_groups_pi["Macro + Core News"]
Xtr_ts, ytr_ts, used_ts, _ = pi_prepare_xy(train_df_fixed, two_stage_feats_pi, main_target_fixed)
Xte_ts, yte_ts, _, test_mask_ts = pi_prepare_xy(test_df_fixed, two_stage_feats_pi, main_target_fixed)
used_ts = [f for f in used_ts if f in Xte_ts.columns]
Xtr_ts, Xte_ts = Xtr_ts[used_ts], Xte_ts[used_ts]
ytr_bin_ts = (ytr_ts > 0).astype(int)
yte_bin_ts = (yte_ts > 0).astype(int)
classifier_names_pi = ["LogisticRegression", "RandomForestClassifier"] + (["XGBoostClassifier"] if XGB_CLASSIFIER_AVAILABLE else [])
stage2_names_pi = ["Ridge", "PoissonRegressor", "RandomForest", "HistGBR_Poisson"]

# 如果训练集中没有两个类别，跳过两阶段分类。
if ytr_bin_ts.nunique() >= 2 and len(Xtr_ts) >= 20:
    pos_mask_tr = ytr_ts > 0
    Xtr_pos, ytr_pos = Xtr_ts.loc[pos_mask_tr], ytr_ts.loc[pos_mask_tr]
    for clf_name in classifier_names_pi:
        try:
            pred_bin, proba, clf = pi_fit_classifier(clf_name, Xtr_ts, ytr_bin_ts, Xte_ts)
            try:
                auc = roc_auc_score(yte_bin_ts, proba) if yte_bin_ts.nunique() >= 2 else np.nan
            except Exception:
                auc = np.nan
            cls_metrics = {
                "Accuracy": accuracy_score(yte_bin_ts, pred_bin),
                "Precision": precision_score(yte_bin_ts, pred_bin, zero_division=0),
                "Recall": recall_score(yte_bin_ts, pred_bin, zero_division=0),
                "F1": f1_score(yte_bin_ts, pred_bin, zero_division=0),
                "ROC_AUC": auc,
            }
            for reg_name in stage2_names_pi:
                try:
                    reg_pred_pos, reg_est = pi_fit_predict_reg(reg_name, Xtr_pos, ytr_pos, Xte_ts)
                    final_pred = np.where(pred_bin == 1, reg_pred_pos, 0.0)
                    metrics = pi_metrics(yte_ts, final_pred)
                    row = {"model_group": "Two-stage", "model_name": f"{clf_name}+{reg_name}", "feature_set": "Macro + Core News", "status": "ok", "n_features": len(used_ts), **cls_metrics, **metrics}
                    two_stage_rows_pi.append(row)
                    comparison_rows_pi.append({k: v for k, v in row.items() if k not in ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]})
                    for t, yt, yp, pb in zip(test_df_fixed.loc[test_mask_ts, time_col_fixed], yte_ts, final_pred, pred_bin):
                        two_stage_pred_rows_pi.append({"model_group": "Two-stage", "model_name": f"{clf_name}+{reg_name}", "week": t, "y_true": yt, "y_pred": yp, "stage1_pred_positive": pb})
                        prediction_rows_pi.append({"model_group": "Two-stage", "model_name": f"{clf_name}+{reg_name}", "week": t, "y_true": yt, "y_pred": yp})
                except Exception as e:
                    two_stage_rows_pi.append({"model_group": "Two-stage", "model_name": f"{clf_name}+{reg_name}", "status": "failed_stage2", "error": str(e)})
        except Exception as e:
            two_stage_rows_pi.append({"model_group": "Two-stage", "model_name": clf_name, "status": "failed_stage1", "error": str(e)})
else:
    two_stage_rows_pi.append({"model_group": "Two-stage", "model_name": "ALL", "status": "skipped_single_class_or_small_sample"})

two_stage_results_pi = pd.DataFrame(two_stage_rows_pi)
two_stage_preds_pi = pd.DataFrame(two_stage_pred_rows_pi)
two_stage_results_pi.to_csv(RESULT_DIR / "two_stage_model_results.csv", index=False, encoding="utf-8-sig")
two_stage_preds_pi.to_csv(RESULT_DIR / "two_stage_prediction_records.csv", index=False, encoding="utf-8-sig")

# ---------- G. 最终改进对比表 ----------
improved_comparison_pi = pd.DataFrame(comparison_rows_pi)
if not improved_comparison_pi.empty:
    ok_imp_pi = improved_comparison_pi[improved_comparison_pi["status"] == "ok"].copy()
else:
    ok_imp_pi = pd.DataFrame()
# Macro Only 基准：取 Macro Only 中 MAE 最低模型。
if not ok_imp_pi.empty and (ok_imp_pi["model_group"] == "Macro Only").any():
    macro_baseline_row_pi = ok_imp_pi[ok_imp_pi["model_group"] == "Macro Only"].sort_values(["MAE", "RMSE"]).iloc[0]
    macro_baseline_mae_pi = macro_baseline_row_pi["MAE"]
    macro_baseline_rmse_pi = macro_baseline_row_pi["RMSE"]
else:
    macro_baseline_row_pi = None
    macro_baseline_mae_pi = np.nan
    macro_baseline_rmse_pi = np.nan

if not improved_comparison_pi.empty:
    improved_comparison_pi["improvement_vs_macro_only_mae_pct"] = (macro_baseline_mae_pi - improved_comparison_pi.get("MAE", np.nan)) / macro_baseline_mae_pi * 100 if pd.notna(macro_baseline_mae_pi) else np.nan
    improved_comparison_pi["improvement_vs_macro_only_rmse_pct"] = (macro_baseline_rmse_pi - improved_comparison_pi.get("RMSE", np.nan)) / macro_baseline_rmse_pi * 100 if pd.notna(macro_baseline_rmse_pi) else np.nan
    improved_comparison_pi = improved_comparison_pi.sort_values(["status", "MAE", "RMSE"], na_position="last").reset_index(drop=True)
improved_comparison_pi.to_csv(RESULT_DIR / "improved_model_comparison.csv", index=False, encoding="utf-8-sig")
prediction_records_pi = pd.DataFrame(prediction_rows_pi)
prediction_records_pi.to_csv(RESULT_DIR / "improved_prediction_records.csv", index=False, encoding="utf-8-sig")

# ---------- H. Expanding-window rolling validation ----------
def pi_train_test_by_year(train_start, train_end, test_start, test_end):
    tr = df_fixed[(df_fixed[time_col_fixed] >= pd.Timestamp(train_start)) & (df_fixed[time_col_fixed] <= pd.Timestamp(train_end))].copy()
    te = df_fixed[(df_fixed[time_col_fixed] >= pd.Timestamp(test_start)) & (df_fixed[time_col_fixed] <= pd.Timestamp(test_end))].copy()
    return tr, te

def pi_eval_simple_group(train_data, test_data, group_name, feats, model_name="RandomForest"):
    Xtr, ytr, used, _ = pi_prepare_xy(train_data, feats, main_target_fixed)
    Xte, yte, _, _ = pi_prepare_xy(test_data, feats, main_target_fixed)
    used = [f for f in used if f in Xte.columns]
    if len(used) == 0 or len(Xtr) < 20 or len(Xte) < 2:
        return {"status": "skipped", "model_group": group_name}
    pred, _ = pi_fit_predict_reg(model_name, Xtr[used], ytr, Xte[used])
    return {"status": "ok", "model_group": group_name, "model_name": model_name, "n_train": len(Xtr), "n_test": len(Xte), **pi_metrics(yte, pred)}

def pi_eval_two_stage_simple(train_data, test_data, feats):
    Xtr, ytr, used, _ = pi_prepare_xy(train_data, feats, main_target_fixed)
    Xte, yte, _, _ = pi_prepare_xy(test_data, feats, main_target_fixed)
    used = [f for f in used if f in Xte.columns]
    if len(used) == 0 or len(Xtr) < 20 or len(Xte) < 2:
        return {"status": "skipped", "model_group": "Two-stage model"}
    Xtr, Xte = Xtr[used], Xte[used]
    ybin = (ytr > 0).astype(int)
    if ybin.nunique() < 2 or (ytr > 0).sum() < 5:
        return {"status": "skipped_single_class", "model_group": "Two-stage model"}
    try:
        pred_bin, proba, clf = pi_fit_classifier("RandomForestClassifier", Xtr, ybin, Xte)
        pred_pos, _ = pi_fit_predict_reg("RandomForest", Xtr.loc[ytr > 0], ytr.loc[ytr > 0], Xte)
        pred = np.where(pred_bin == 1, pred_pos, 0.0)
        return {"status": "ok", "model_group": "Two-stage model", "model_name": "RFClassifier+RFRegressor", "n_train": len(Xtr), "n_test": len(Xte), **pi_metrics(yte, pred)}
    except Exception as e:
        return {"status": "failed", "model_group": "Two-stage model", "error": str(e)}

rolling_windows_pi = [
    ("2022-2023_train_2024_test", "2022-01-01", "2023-12-31", "2024-01-01", "2024-12-31"),
    ("2022-2024_train_2025_test", "2022-01-01", "2024-12-31", "2025-01-01", "2025-12-31"),
    ("2022-2025_train_2026_test", "2022-01-01", "2025-12-31", "2026-01-01", "2026-12-31"),
]
rolling_rows_pi = []
for win_name, tr_s, tr_e, te_s, te_e in rolling_windows_pi:
    tr, te = pi_train_test_by_year(tr_s, tr_e, te_s, te_e)
    if len(tr) < 20 or len(te) < 2:
        rolling_rows_pi.append({"window": win_name, "model_group": "ALL", "status": f"skipped_small_sample_train={len(tr)}_test={len(te)}"})
        continue
    groups_for_roll = {
        "Macro Only": macro_only_pi,
        "Macro + Full News": macro_full_news_pi,
        "Macro + Core News": feature_groups_pi["Macro + Core News"],
        "Macro + Selected News": feature_groups_pi["Macro + Selected News"],
    }
    for group_name, feats in groups_for_roll.items():
        row = pi_eval_simple_group(tr, te, group_name, feats, model_name="RandomForest")
        row["window"] = win_name
        rolling_rows_pi.append(row)
    row = pi_eval_two_stage_simple(tr, te, feature_groups_pi["Macro + Core News"])
    row["window"] = win_name
    rolling_rows_pi.append(row)

rolling_detail_pi = pd.DataFrame(rolling_rows_pi)
rolling_ok_pi = rolling_detail_pi[rolling_detail_pi.get("status", pd.Series(dtype=str)) == "ok"].copy() if not rolling_detail_pi.empty else pd.DataFrame()
if not rolling_ok_pi.empty:
    rolling_avg_pi = rolling_ok_pi.groupby("model_group", as_index=False).agg({
        "MAE": "mean", "RMSE": "mean", "sMAPE": "mean", "Directional_Accuracy": "mean"
    })
    rolling_avg_pi.insert(0, "window", "average")
    rolling_output_pi = pd.concat([rolling_detail_pi, rolling_avg_pi], ignore_index=True, sort=False)
else:
    rolling_output_pi = rolling_detail_pi
rolling_output_pi.to_csv(RESULT_DIR / "rolling_validation_improved_results.csv", index=False, encoding="utf-8-sig")

# ---------- I. 误差诊断 ----------
# 选择最终改进表中 MAE 最好的非 Macro Only 模型；若没有，则用全表最佳。
if not ok_imp_pi.empty:
    non_macro_ok = ok_imp_pi[ok_imp_pi["model_group"] != "Macro Only"].copy()
    error_model_row_pi = (non_macro_ok if not non_macro_ok.empty else ok_imp_pi).sort_values(["MAE", "RMSE"]).iloc[0]
    err_pred = prediction_records_pi[(prediction_records_pi["model_group"] == error_model_row_pi["model_group"]) & (prediction_records_pi["model_name"] == error_model_row_pi["model_name"])].copy()
else:
    error_model_row_pi = None
    err_pred = pd.DataFrame()

if not err_pred.empty:
    err_pred["week"] = pd.to_datetime(err_pred["week"])
    err_pred["absolute_error"] = (err_pred["y_true"] - err_pred["y_pred"]).abs()
    diag_cols = [time_col_fixed, "weekly_layoff_news_count", "weekly_layoff_news_share", "weekly_negative_share"] + macro_features_fixed[:8]
    diag_cols = [c for c in diag_cols if c in test_df_fixed.columns]
    diag_df = test_df_fixed[diag_cols].copy().rename(columns={time_col_fixed: "week"})
    diag_df["week"] = pd.to_datetime(diag_df["week"])
    top_errors_pi = err_pred.merge(diag_df, on="week", how="left").sort_values("absolute_error", ascending=False).head(10)
else:
    top_errors_pi = pd.DataFrame(columns=["week", "y_true", "y_pred", "absolute_error", "weekly_layoff_news_count", "weekly_layoff_news_share", "weekly_negative_share"])
top_errors_pi.to_csv(RESULT_DIR / "top_prediction_errors.csv", index=False, encoding="utf-8-sig")

# ---------- J. 图表 ----------
if not ok_imp_pi.empty:
    plot_imp = ok_imp_pi.sort_values("MAE").head(20).copy()
    plot_imp["label"] = plot_imp["model_group"] + "\n" + plot_imp["model_name"]
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    ax[0].barh(plot_imp["label"], plot_imp["MAE"])
    ax[0].set_title("Improved models: MAE")
    ax[0].set_xlabel("MAE")
    ax[1].barh(plot_imp["label"], plot_imp["RMSE"])
    ax[1].set_title("Improved models: RMSE")
    ax[1].set_xlabel("RMSE")
    plot_save_close(FIG_DIR / "improved_model_comparison_mae_rmse.png")

if not two_stage_preds_pi.empty:
    # 画两阶段 MAE 最佳模型。
    best_ts = two_stage_results_pi[two_stage_results_pi["status"] == "ok"].sort_values(["MAE", "RMSE"]).iloc[0] if (two_stage_results_pi.get("status", pd.Series(dtype=str)) == "ok").any() else None
    if best_ts is not None:
        ts_plot = two_stage_preds_pi[two_stage_preds_pi["model_name"] == best_ts["model_name"]].copy()
        plt.figure(figsize=(10, 5))
        plt.plot(pd.to_datetime(ts_plot["week"]), ts_plot["y_true"], marker="o", label="Actual")
        plt.plot(pd.to_datetime(ts_plot["week"]), ts_plot["y_pred"], marker="o", label="Two-stage predicted")
        plt.title(f"Two-stage prediction vs actual: {best_ts['model_name']}")
        plt.xlabel("Week")
        plt.ylabel("Next-week layoff event count")
        plt.legend()
        plot_save_close(FIG_DIR / "two_stage_prediction_vs_actual.png")

if not rolling_ok_pi.empty:
    roll_plot = rolling_ok_pi.copy()
    plt.figure(figsize=(11, 5))
    for group in roll_plot["model_group"].unique():
        sub = roll_plot[roll_plot["model_group"] == group]
        plt.plot(sub["window"], sub["MAE"], marker="o", label=group)
    plt.xticks(rotation=25, ha="right")
    plt.ylabel("MAE")
    plt.title("Rolling validation comparison")
    plt.legend(fontsize=8)
    plot_save_close(FIG_DIR / "rolling_validation_comparison.png")

if not top_errors_pi.empty:
    top_plot = top_errors_pi.sort_values("absolute_error")
    plt.figure(figsize=(10, 5))
    plt.barh(top_plot["week"].dt.strftime("%Y-%m-%d"), top_plot["absolute_error"])
    plt.xlabel("Absolute error")
    plt.title("Top prediction error weeks")
    plot_save_close(FIG_DIR / "top_error_weeks.png")

# ---------- K. 文字解释 ----------
# 误差诊断文字。
if not top_errors_pi.empty:
    max_err = top_errors_pi.iloc[0]
    err_text = f"最大误差周为 {pd.to_datetime(max_err['week']).date()}，真实值={max_err['y_true']:.3f}，预测值={max_err['y_pred']:.3f}，绝对误差={max_err['absolute_error']:.3f}。"
else:
    err_text = "没有生成有效误差诊断样本。"
safe_write_text(TEXT_DIR / "prediction_error_diagnosis.md", f"""# 预测误差诊断

本部分基于改进模型中 MAE 最低的非 Macro Only 模型，找出测试集中误差最大的前 10 周。{err_text}

这些大误差周通常需要结合当周裁员新闻强度、负面新闻比例和宏观变量变化解释。若大误差集中在突发裁员周，说明周度新闻与宏观变量仍难以完全捕捉公司层面的突然决策。
""")

# 汇总判断。
if not ok_imp_pi.empty and macro_baseline_row_pi is not None:
    best_overall_pi = ok_imp_pi.sort_values(["MAE", "RMSE"]).iloc[0]
    best_non_macro_pi = (ok_imp_pi[ok_imp_pi["model_group"] != "Macro Only"].sort_values(["MAE", "RMSE"]).iloc[0] if (ok_imp_pi["model_group"] != "Macro Only").any() else None)
    if best_non_macro_pi is not None:
        non_macro_mae_imp = (macro_baseline_mae_pi - best_non_macro_pi["MAE"]) / macro_baseline_mae_pi * 100
        non_macro_rmse_imp = (macro_baseline_rmse_pi - best_non_macro_pi["RMSE"]) / macro_baseline_rmse_pi * 100
        improve_sentence = f"最佳非 Macro Only 改进模型为 `{best_non_macro_pi['model_group']}` / `{best_non_macro_pi['model_name']}`，相对 Macro Only 最佳模型 MAE 改善 {non_macro_mae_imp:.2f}%，RMSE 改善 {non_macro_rmse_imp:.2f}%。"
    else:
        improve_sentence = "没有有效的非 Macro Only 改进模型。"
    best_sentence = f"整体 MAE 最低模型为 `{best_overall_pi['model_group']}` / `{best_overall_pi['model_name']}`，MAE={best_overall_pi['MAE']:.3f}，RMSE={best_overall_pi['RMSE']:.3f}。Macro Only 基准为 `{macro_baseline_row_pi['model_name']}`，MAE={macro_baseline_mae_pi:.3f}，RMSE={macro_baseline_rmse_pi:.3f}。"
else:
    best_sentence = "没有有效模型比较结果。"
    improve_sentence = "无法判断改进幅度。"

# 两阶段结论。
if (two_stage_results_pi.get("status", pd.Series(dtype=str)) == "ok").any():
    best_ts_sum = two_stage_results_pi[two_stage_results_pi["status"] == "ok"].sort_values(["MAE", "RMSE"]).iloc[0]
    two_stage_sentence = f"两阶段最佳模型为 `{best_ts_sum['model_name']}`，MAE={best_ts_sum['MAE']:.3f}，RMSE={best_ts_sum['RMSE']:.3f}，分类 F1={best_ts_sum.get('F1', np.nan):.3f}。"
else:
    two_stage_sentence = "两阶段模型未得到有效结果或未改善。"

# rolling validation 结论。
if not rolling_ok_pi.empty:
    roll_avg = rolling_output_pi[rolling_output_pi["window"] == "average"].sort_values("MAE") if "window" in rolling_output_pi.columns else pd.DataFrame()
    if not roll_avg.empty:
        roll_sentence = f"滚动验证平均 MAE 最低的是 `{roll_avg.iloc[0]['model_group']}`，平均 MAE={roll_avg.iloc[0]['MAE']:.3f}，RMSE={roll_avg.iloc[0]['RMSE']:.3f}。"
    else:
        roll_sentence = "滚动验证已生成逐窗口结果，但平均结果不足。"
else:
    roll_sentence = "滚动验证没有足够有效窗口。"

summary_text_pi = f"""# Prediction Improvement 结果摘要

## 1. 原始 Macro + News 为什么可能表现差

修复后的 Full News 特征数较多，新闻变量之间相关性强，且周度样本量有限。把全部新闻特征直接塞入模型容易增加噪声、共线性和过拟合风险。因此本章节新增 `Macro + Core News`、`Macro + CorrTop News`、`Macro + LassoSelected News`、`Macro + RFTop News` 和 `Macro + Selected News`，并且所有特征筛选都只在训练集内完成。

## 2. 特征筛选后是否改善

{best_sentence}

{improve_sentence}

如果改善幅度为负或很小，说明筛选后的新闻变量仍未提供稳定预测增量；此时更合理的结论是新闻变量具有机制解释价值，但预测增量有限。

## 3. 两阶段模型是否改善

{two_stage_sentence}

两阶段模型先判断下一周是否有裁员事件，再预测正裁员周数量。该结构更贴近零值较多的计数目标，但如果分类阶段不能稳定识别正裁员周，最终预测仍可能不优于简单 Macro Only。

## 4. 计数模型是否改善

本章节新增 PoissonRegressor 调参、TweedieRegressor、HistGradientBoostingRegressor(loss='poisson')，以及可用时的 XGBoost count:poisson。计数模型更符合非负事件数目标，但其表现仍取决于特征信号强度和样本规模。

## 5. Rolling validation 下结论是否稳定

{roll_sentence}

滚动验证用于检查不同年份测试期下结论是否稳定。如果某类新闻增强模型只在单一测试期改善，则不应写成稳定预测优势。

## 6. 报告建议表述

若最终结果显示 Macro + News 或筛选后新闻模型仍不如 Macro Only，应如实写为：新闻变量在 TLCC 和滞后回归中体现了机制解释价值或前瞻关联，但在严格样本外预测中，增量预测价值有限且不稳定。不要为了提升结果删除零裁员周，也不要使用测试集做特征筛选。
"""
safe_write_text(TEXT_DIR / "improved_prediction_results_summary.md", summary_text_pi)

print("Prediction Improvement 完成。")
print("Macro Only baseline:", None if macro_baseline_row_pi is None else macro_baseline_row_pi[["model_group", "model_name", "MAE", "RMSE"]].to_dict())
if not ok_imp_pi.empty:
    print("Top improved comparison rows:")
    display(improved_comparison_pi.head(20))
print(summary_text_pi)


开始 Prediction Improvement 章节...
新增 log1p 新闻数量特征： ['log1p_weekly_news_count', 'log1p_weekly_news_count_lag1', 'log1p_weekly_news_count_lag2', 'log1p_weekly_news_count_lag3', 'log1p_weekly_news_count_lag4', 'log1p_weekly_layoff_news_count', 'log1p_weekly_layoff_news_count_lag1', 'log1p_weekly_layoff_news_count_lag2', 'log1p_weekly_layoff_news_count_lag3', 'log1p_weekly_layoff_news_count_lag4', 'log1p_weekly_hiring_news_count', 'log1p_weekly_hiring_news_count_lag1', 'log1p_weekly_hiring_news_count_lag2', 'log1p_weekly_hiring_news_count_lag3', 'log1p_weekly_hiring_news_count_lag4']
缺失的核心新闻变量： []
      selection_method  n_features                                                                                                                                                                                                                                                                                                                                                                               